# Pipeline OCR — Partie 1 — V14.0
### Rotation de toutes les pages · blocs anti-décalage (Titre de travail + Engagement) · RAW auditable

**Périmètre inchangé** : PDF → rendu → orientation → classification → extraction Qwen → *une* relecture si nécessaire → JSON RAW par dossier.
Aucune normalisation, aucune règle métier, aucun rapprochement entre documents : tout cela reste en Partie 2.
Contrat `DOM_EXTRACTION_V1`, **99 champs**, même hash de schéma que V13.8.7.

| Sujet | V13.8.7 | V14.0 |
|---|---|---|
| Rotation | Qwen, pages TTR seulement, **après** classification, sans contrôle | Qwen sur **toutes** les pages, **avant** classification ; rotation sans perte ; re-contrôle des pages tournées (rattrape 90↔270) ; contrôle déterministe de l'axe du texte ; deskew (2°–15°) |
| Décalage TTR | structure POSTE → DURÉE → DU → AU → LIEU | **conservée mot pour mot** + contrôles croisés gratuits (champ = date réellement observée) |
| Décalage Engagement | une phrase générique | bloc structurel « observer puis associer » + objet d'audit `_DOM_EVIDENCE` + validation de type de chaque champ |
| Fusion initial / relecture | la relecture écrasait toute valeur différente | un bloc décalable est arbitré **lecture entière contre lecture entière** ; deux valeurs valides différentes ⇒ désaccord tracé, jamais d'écrasement |
| N° de permis TTR | regex = une moitié **ou** l'autre ; la structure complète déclenchait un recovery sans raison tracée | structure complète `NN-NNNNNNNN / NN-NN-NNNNNN` (interrupteur `TTR_PERMIT_EXPECT_FULL_STRUCTURE`) |
| Statut des pages TTR | toujours `PARTIELLE` / `LOW` (2 champs critiques jamais extraits) | critiques = champs actifs ; `ttr7_final_valid` pilote réellement statut et drapeaux |
| Recovery « 2400 px » | rogné à ≈1888 px par `MAX_PIXELS` | réel (plafond relevé ; images initiales inchangées) |
| Mesure | `tokens_out` faux en batch ; CSV profiler écrasés à chaque PDF | tokens réels, préfill / décodage séparés, CSV par dossier, orientation comptée |

**Non modifié volontairement (baseline qualité)** : modèle et chargement, résolutions initiales 1100 / 1400 / 1800, prompts classification / contrat / contrat spécifique (identiques à l'octet près), profil TTR-7, PTR sans appel Qwen, `repetition_penalty=1.0`, batching.

## V14.1 — baseline production nettoyée

Comportement fonctionnel conservé : orientation, deskew, classification, extraction structurelle DOM/TTR, recovery unique et audit.

Nettoyage : fonctions mortes supprimées, libellés legacy retirés et sorties isolées en V14.1. Aucune réduction de résolution ni de contrôle qualité n'est activée par défaut.


## 1. Dépendances

In [ ]:
# Environnement neuf Domino :
# %pip install -q -U 'transformers>=5.9' accelerate pymupdf pillow pandas psutil opencv-python-headless
#
# Recommandé (fast path des 48 couches Gated DeltaNet de Qwen3.6-27B) — la cellule 4 signale leur absence :
# %pip install -q causal-conv1d flash-linear-attention
# Après installation : rejouer le corpus de non-régression avant mise en production.

## 2. Imports

In [ ]:
import gc
import hashlib
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import cv2
import fitz
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

print('Python :', sys.version.split()[0], '| Torch :', torch.__version__,
      '| GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun')

## 3. Configuration
Tous les interrupteurs d'A/B sont ici : `BATCH_TTR_WITH_STANDARD`, `TTR_PERMIT_EXPECT_FULL_STRUCTURE`, `DESKEW_ENABLED`, `ORIENTATION_VERIFY_ROTATED`, `SKIP_BLANK_PAGES`.

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---------- Contrat de données (INCHANGÉ : 99 champs, même hash) ----------
SCHEMA_VERSION = 'DOM_EXTRACTION_V1'
PIPELINE_VERSION = 'GENERIC_V14_1_PART1_CLEAN_PRODUCTION'
FIELD_SCHEMA_HASH = 'da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e'

# ---------- Exécution ----------
MAX_PDFS = None     # production : tous les PDF ; mettre 10 pour un test
RESUME = True       # un JSON d'une autre pipeline_version n'est PAS repris

# ---------- Images (résolutions initiales IDENTIQUES à V13.8.7) ----------
PDF_ZOOM = 2.0
IMAGE_MAX_SIZE = 1400
IMAGE_MAX_SIZE_CLASSIFICATION = 1100
PDF_ZOOM_HAUTE_DEF = 4
IMAGE_MAX_SIZE_HAUTE_DEF = 1800
IMAGE_MAX_SIZE_RECOVERY = 2400
MIN_PIXELS = 4 * 32 * 32
# V13.8.7 plafonnait à 2400*32*32 (≈2,46 Mpx) : une page A4 « 2400 px » était
# silencieusement ramenée à ≈1888 px, soit presque la résolution initiale du TTR.
# 4096*32*32 (≈4,19 Mpx) rend le recovery 2400 px RÉEL. Les images 1100/1400/1800
# étaient déjà sous l'ancien plafond : elles sont strictement inchangées.
MAX_PIXELS = 4096 * 32 * 32

# ---------- Orientation (TOUTES les pages, AVANT classification) ----------
ORIENTATION_ENABLED = True
ORIENTATION_MAX_NEW_TOKENS = 40
ORIENTATION_VERIFY_ROTATED = True      # re-contrôle Qwen des seules pages tournées
ORIENTATION_MAX_VERIFY_PASSES = 2
ORIENTATION_AXIS_RATIO = 2.0           # contrôle déterministe : marge de décision
ORIENTATION_AXIS_MIN_EVIDENCE = 600

# ---------- Deskew (petites inclinaisons, après rotation 90/180/270) ----------
DESKEW_ENABLED = True
DESKEW_MIN_ABS_DEG = 2.0    # < 2° : on ne touche pas à l'image (baseline préservée)
DESKEW_MAX_ABS_DEG = 15.0
DESKEW_MIN_LINES = 6
DESKEW_MAX_MAD_DEG = 1.8

# ---------- Classification ----------
CLASSIFICATION_THRESHOLD = 0.90
CLASSIFICATION_RETRY_ON_AUTRE = True
CLASSIFICATION_RETRY_LOW_CONFIDENCE = True
CLASSIFICATION_HARD_MIN_CONFIDENCE = 0.90
BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT = True
SKIP_BLANK_PAGES = True
BLANK_MAX_DARK_RATIO = 0.0004   # part de pixels < 128 ; très conservateur

# ---------- Batch ----------
GPU_BATCH_SIZE_CLASSIFICATION = 16
GPU_BATCH_SIZE_EXTRACTION_STANDARD = 4
GPU_BATCH_SIZE_EXTRACTION_HD = 2
# False = comportement V13.8.7 (TTR décodé seul). True = TTR dans le même
# model.generate que DOM/CTR/CTS (≈ -330 pas de décodage séquentiels).
# À n'activer qu'après le benchmark A/B champ par champ.
BATCH_TTR_WITH_STANDARD = False

# ---------- Génération (plafonds = garde-fous ; l'EOS arrête avant) ----------
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
MAX_NEW_TOKENS_RECOVERY = 1900
MAX_NEW_TOKENS_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 1100,   # +200 : objet d'audit _DOM_EVIDENCE
    'CONTRAT_TRAVAIL': 1300,
    'CONTRAT_SPECIFIQUE': 1500,
    'TITRE_TRAVAIL': 700,
}
MAX_NEW_TOKENS_RECOVERY_BY_DOC = {
    'ENGAGEMENT_DOMICILIATION': 1300,
    'CONTRAT_TRAVAIL': 1400,
    'CONTRAT_SPECIFIQUE': 1600,
    'TITRE_TRAVAIL': 700,
}
COMPACT_OUTPUT_ENABLED = True

# ---------- Recovery : UNE relecture page entière 2400 px au maximum ----------
SEUIL_REMPLISSAGE_MIN = 0.40
ENABLE_PAGE_RECOVERY = True
RECOVERY_ON_CRITICAL_MISSING = True
RECOVERY_ON_LOW_FILL = True
# Une incompatibilité de TYPE (date dans un lieu, texte dans une date...) est le
# symptôme n°1 d'un décalage vertical. Elle déclenche la relecture uniquement si
# elle touche un champ critique ou un champ d'un bloc sensible au décalage.
RECOVERY_ON_SEMANTIC_MISMATCH = True
RECOVERY_ON_SHIFT_BLOCK_INCOHERENT = True
# Après relecture, si un bloc décalable reste incohérent : les seuls champs du bloc
# dont le TYPE est incompatible passent à null (valeur lue conservée dans
# `nullified_values`). Jamais de permutation automatique.
NULLIFY_TYPE_INCOMPATIBLE_IN_UNRESOLVED_BLOCK = True

# ---------- TTR : numéro de permis ----------
# True  : TTR_NUMERO_PERMIS = structure COMPLÈTE « NN-NNNNNNNN / NN-NN-NNNNNN »
#         (cohérent avec CTR/CTS_NUMERO_PERMIS_TRAVAIL).
# False : ancien comportement V13.8.7 (une seule des deux références).
TTR_PERMIT_EXPECT_FULL_STRUCTURE = True

# ---------- Confidence opérationnelle (pas une log-prob Qwen) ----------
CONFIDENCE_METHOD = 'OPERATIONAL_CONSENSUS_SEMANTIC_V1'
CONFIDENCE_HIGH = 90
CONFIDENCE_MEDIUM = 75

# ---------- Audit ----------
STORE_QWEN_RAW_TEXT_IN_ATTEMPTS = True
STORE_PARSED_DATA_IN_ATTEMPTS = False
PRINT_CALL_DIAGNOSTICS = True
PROFILER_ENABLED = True
PROFILE_PREFILL = True      # horodate le 1er token : sépare préfill / décodage

# ---------- Fichiers ----------
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_ROOT = Path('/mnt/data/document_pipeline/V14_1')
RAW_ROOT = OUTPUT_ROOT / '01_extraction_raw'
JSON_DIR = RAW_ROOT / 'json_dossiers'
PROFILER_DIR = RAW_ROOT / 'profiler'
LOG_PATH = RAW_ROOT / 'pipeline_extraction_v14_1.log'
MANIFEST_PATH = RAW_ROOT / 'extraction_manifest_v14_1.json'
INDEX_CSV_PATH = RAW_ROOT / 'extraction_index_v14_1.csv'

for _d in (INPUT_DIR, JSON_DIR, PROFILER_DIR):
    _d.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob('*.pdf'))
if MAX_PDFS is not None:
    pdfs = pdfs[:int(MAX_PDFS)]

print('Pipeline          :', PIPELINE_VERSION)
print('Schema            :', SCHEMA_VERSION, '|', FIELD_SCHEMA_HASH[:16] + '…')
print('PDFs sélectionnés :', len(pdfs))
print('Orientation       : toutes les pages, avant classification')
print('Recovery          : unique, page entière', IMAGE_MAX_SIZE_RECOVERY, 'px')
print('Permis TTR        :', 'structure complète A / B' if TTR_PERMIT_EXPECT_FULL_STRUCTURE else 'demi-référence (legacy)')

## 4. Chargement Qwen (identique à V13.8.7) + contrôles d'environnement

In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Ce pipeline nécessite un GPU CUDA.')

torch.backends.cuda.matmul.allow_tf32 = True

# --- Contrôle du fast path Gated DeltaNet (48 des 64 couches de Qwen3.6-27B) ---
# Sans ces deux paquets, Transformers utilise une implémentation PyTorch de
# référence, correcte mais nettement plus lente. On ne fait que le SIGNALER.
import importlib.util as _ilu
_fla = _ilu.find_spec('fla') is not None
_cc1d = _ilu.find_spec('causal_conv1d') is not None
print('flash-linear-attention :', 'OK' if _fla else 'ABSENT')
print('causal-conv1d          :', 'OK' if _cc1d else 'ABSENT')
if not (_fla and _cc1d):
    print('⚠️ Fast path DeltaNet probablement inactif -> préfill/décodage ralentis.')
    print('   Après installation, rejouer le corpus de non-régression (numérique différente).')

print('Chargement du processor...')
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH, trust_remote_code=True,
    min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = 'left'
_ip = getattr(processor, 'image_processor', None)
print('Image processor size   :', getattr(_ip, 'size', None),
      '| max_pixels =', getattr(_ip, 'max_pixels', None))
print('   -> vérifier que le plafond vaut bien', MAX_PIXELS, 'px ; sinon le recovery 2400 est rogné.')

try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

# Chargement IDENTIQUE à V13.8.7 (poids FP8 déquantifiés -> calcul BF16).
# C'est la baseline qualité. Le FP8 natif / vLLM change la numérique : benchmark d'abord.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

# Identifiants de fin de séquence (pour compter les VRAIS tokens générés en batch).
_eos = set()
for _src in (getattr(model, 'generation_config', None), processor.tokenizer):
    _v = getattr(_src, 'eos_token_id', None)
    if isinstance(_v, (list, tuple)): _eos.update(int(x) for x in _v)
    elif _v is not None: _eos.add(int(_v))
if processor.tokenizer.pad_token_id is not None:
    _eos.add(int(processor.tokenizer.pad_token_id))
EOS_TOKEN_IDS = _eos

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s | VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB')
_device_map = getattr(model, 'hf_device_map', None)
if _device_map and any(str(v).lower() in {'cpu','disk'} for v in _device_map.values()):
    print('⚠️ Offload CPU/disk détecté : inférence fortement ralentie.')

## 5. Images : rendu en cache, rotation, deskew, contrôle d'axe
Toute image envoyée à Qwen passe par `page_image()` : rendu → rotation 0/90/180/270 **sans perte** → deskew éventuel **à pleine résolution** → réduction LANCZOS.
Pour une page droite, l'image est identique au pixel près à celle de V13.8.7.

In [ ]:
# =====================================================================
# Utilitaires fichiers / JSON
# =====================================================================
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(line + '\n')


def parse_json_response(text):
    if not text:
        return {}
    clean = str(text).strip()
    clean = re.sub(r'^```(?:json)?', '', clean, flags=re.I).strip()
    clean = re.sub(r'```$', '', clean).strip()
    match = re.search(r'\{.*\}', clean, flags=re.S)
    if not match:
        return {}
    candidate = match.group(0)
    for attempt in (candidate, re.sub(r',\s*([}\]])', r'\1', candidate)):
        try:
            obj = json.loads(attempt)
            return obj if isinstance(obj, dict) else {}
        except Exception:
            pass
    return {}


# =====================================================================
# Rendu PDF — UN seul rendu par (page, zoom) et par dossier
# =====================================================================
_RENDER_CACHE = {}

def render_cache_clear():
    _RENDER_CACHE.clear()


def _render_raw(pdf_path, page_index, zoom):
    """Rendu brut non tourné, mis en cache : le recovery ne re-rend pas la page."""
    key = (str(pdf_path), int(page_index), float(zoom))
    img = _RENDER_CACHE.get(key)
    if img is None:
        doc = fitz.open(str(pdf_path))
        try:
            pix = doc.load_page(int(page_index)).get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
            img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
        finally:
            doc.close()
        _RENDER_CACHE[key] = img
    return img


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.asarray(image.convert('L'))
    return float((arr > 245).sum() / arr.size)


def dark_ratio(image):
    arr = np.asarray(image.convert('L'))
    return float((arr < 128).sum() / arr.size)


# =====================================================================
# Rotation 0/90/180/270 — SANS PERTE (transposition de pixels)
# =====================================================================
_TRANSPOSE = {90: Image.ROTATE_270, 180: Image.ROTATE_180, 270: Image.ROTATE_90}

def rotate_clockwise(img, angle):
    a = int(angle or 0) % 360
    return img.transpose(_TRANSPOSE[a]) if a in _TRANSPOSE else img


# =====================================================================
# Contrôle DÉTERMINISTE de l'axe des lignes de texte
# =====================================================================
# Distingue {0,180} de {90,270}. Ne distingue PAS 0 de 180 : ce contrôle ne
# remplace pas Qwen, il détecte ses erreurs d'un quart de tour.
# Calibré sur 11 pages réelles x 4 rotations : 40 verdicts justes, 4 'UNKNOWN', 0 faux.
def text_axis(img, long_side=900):
    g = np.asarray(img.convert('L')); h, w = g.shape
    s = long_side / max(h, w)
    if s < 1:
        g = cv2.resize(g, (max(1, int(w*s)), max(1, int(h*s))), interpolation=cv2.INTER_AREA)
    bw = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, 31, 15)

    def _score(b):
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
        d = cv2.morphologyEx(b, cv2.MORPH_CLOSE, k)
        n, _, st, _ = cv2.connectedComponentsWithStats(d, 8)
        total = 0
        for i in range(1, n):
            ww, hh = st[i, cv2.CC_STAT_WIDTH], st[i, cv2.CC_STAT_HEIGHT]
            if ww >= 40 and 4 <= hh <= 28 and ww / hh >= 5:
                total += int(ww)
        return total

    sh = _score(bw); sv = _score(cv2.rotate(bw, cv2.ROTATE_90_CLOCKWISE))
    if max(sh, sv) < ORIENTATION_AXIS_MIN_EVIDENCE:
        axis = 'UNKNOWN'
    elif sh >= ORIENTATION_AXIS_RATIO * sv:
        axis = 'HORIZONTAL'
    elif sv >= ORIENTATION_AXIS_RATIO * sh:
        axis = 'VERTICAL'
    else:
        axis = 'UNKNOWN'
    return {'axis': axis, 'score_horizontal': sh, 'score_vertical': sv}


# =====================================================================
# Deskew — petites inclinaisons (|angle| entre DESKEW_MIN et DESKEW_MAX)
# =====================================================================
# NB : la fonction deskew_for_vlm de V13.x n'était jamais appelée ET corrigeait
# dans le mauvais sens (elle doublait l'inclinaison). Signe corrigé et testé ici.
def estimate_small_skew_deg(img):
    """Angle médian des lignes quasi horizontales, repère image (y vers le bas).
    > 0 : le contenu penche dans le sens HORAIRE. None : preuve insuffisante."""
    gray = cv2.cvtColor(np.asarray(img.convert('RGB')), cv2.COLOR_RGB2GRAY)
    h, w = gray.shape[:2]
    scale = min(1.0, 1600.0 / max(h, w))
    if scale < 1.0:
        gray = cv2.resize(gray, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), 50, 150, apertureSize=3)
    min_len = max(80, int(gray.shape[1] * 0.12))
    lines = cv2.HoughLinesP(edges, 1, np.pi/1800, threshold=60, minLineLength=min_len, maxLineGap=20)
    if lines is None:
        return None, 0
    angles = []
    # OpenCV renvoie (N,1,4) ou (N,4) selon la version : on aplatit dans les deux cas.
    for x1, y1, x2, y2 in np.asarray(lines, dtype=float).reshape(-1, 4):
        if abs(x2 - x1) < 1:
            continue
        a = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if -DESKEW_MAX_ABS_DEG <= a <= DESKEW_MAX_ABS_DEG:
            angles.append(a)
    if len(angles) < DESKEW_MIN_LINES:
        return None, len(angles)
    med = float(np.median(angles))
    mad = float(np.median(np.abs(np.asarray(angles) - med)))
    if mad > DESKEW_MAX_MAD_DEG:
        return None, len(angles)
    return med, len(angles)


def rotate_free_white(img, angle_ccw_deg):
    """Rotation libre, sens ANTI-horaire positif (convention OpenCV), fond blanc."""
    arr = np.asarray(img.convert('RGB')); h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w/2.0, h/2.0), angle_ccw_deg, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(h*sin + w*cos), int(h*cos + w*sin)
    M[0, 2] += nw/2 - w/2.0; M[1, 2] += nh/2 - h/2.0
    rot = cv2.warpAffine(arr, M, (nw, nh), flags=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))
    return Image.fromarray(rot)


def measure_deskew(img):
    meta = {'deskew_enabled': bool(DESKEW_ENABLED), 'deskew_applied': False,
            'deskew_detected_angle_deg': None, 'deskew_correction_ccw_deg': 0.0,
            'deskew_evidence_lines': 0}
    if not DESKEW_ENABLED:
        return meta
    angle, n = estimate_small_skew_deg(img)
    meta['deskew_evidence_lines'] = int(n)
    if angle is None:
        return meta
    meta['deskew_detected_angle_deg'] = round(float(angle), 3)
    if DESKEW_MIN_ABS_DEG <= abs(angle) <= DESKEW_MAX_ABS_DEG:
        meta['deskew_applied'] = True
        # contenu penché de +a (horaire) -> rotation anti-horaire de +a
        meta['deskew_correction_ccw_deg'] = round(float(angle), 3)
    return meta


# =====================================================================
# POINT D'ENTRÉE UNIQUE des images envoyées à Qwen
# =====================================================================
def page_image(pdf_path, page_like, max_side, zoom):
    """rendu (cache) -> rotation 90/180/270 sans perte -> deskew à pleine
    résolution -> réduction LANCZOS. Pour une page droite et non inclinée, l'image
    est identique au pixel près à celle de V13.8.7."""
    idx = int(page_like.get('index', int(page_like.get('page_num', 1)) - 1))
    img = _render_raw(pdf_path, idx, zoom)
    img = rotate_clockwise(img, page_like.get('rotation_clockwise', 0))
    corr = float(page_like.get('deskew_correction_ccw_deg', 0.0) or 0.0)
    if corr:
        img = rotate_free_white(img, corr)
    return resize_image(img, max_side=max_side)


def standard_image(pdf_path, page_like):
    return page_image(pdf_path, page_like, IMAGE_MAX_SIZE, PDF_ZOOM)


def image_for_classification(image):
    return resize_image(image, max_side=IMAGE_MAX_SIZE_CLASSIFICATION)


def pdf_to_pages(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise ValueError(f'PDF absent ou vide : {path}')
    doc = fitz.open(str(path))
    try:
        n = doc.page_count
    finally:
        doc.close()
    pages = []
    for i in range(n):
        page = {'index': i, 'page_num': i + 1, 'rotation_clockwise': 0,
                'deskew_correction_ccw_deg': 0.0}
        img = standard_image(path, page)
        page.update({'image': img, 'width': img.width, 'height': img.height,
                     'white_ratio': round(white_ratio(img), 6),
                     'dark_ratio': round(dark_ratio(img), 6)})
        pages.append(page)
    return pages


def is_missing_raw(value):
    # Sert uniquement à décider d'une relecture. Ne normalise jamais le RAW.
    if value is None:
        return True
    if isinstance(value, str):
        t = value.strip()
        return (not t) or t.upper() in {'NULL', 'NONE', 'N/A', 'NA', 'ILLISIBLE', 'NON LISIBLE'}
    if isinstance(value, (list, dict)):
        return len(value) == 0
    return False


def taux_remplissage(data, champs):
    if not champs:
        return 1.0
    n = sum(1 for c in champs if not is_missing_raw((data or {}).get(c)))
    return round(n / len(champs), 4)

print('✅ Utilitaires image V14 : rendu en cache, rotation sans perte, deskew corrigé, contrôle d’axe')

## 6. Profiler

In [ ]:
from contextlib import contextmanager

PROFILER_EVENTS = []
PAGE_PROFILER_ROWS = []
PROFILER_T0 = None

def _pnow():
    return time.perf_counter()

def _psync():
    try:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass

def _pelapsed():
    return 0.0 if PROFILER_T0 is None else _pnow() - PROFILER_T0

def _pprint(msg):
    if PROFILER_ENABLED:
        print(f"[PROF {_pelapsed():8.3f}s] {msg}", flush=True)

def profiler_reset():
    global PROFILER_EVENTS, PAGE_PROFILER_ROWS, PROFILER_T0
    PROFILER_EVENTS = []; PAGE_PROFILER_ROWS = []; PROFILER_T0 = _pnow()
    _pprint('START DOSSIER')

def pevent(stage, seconds, **meta):
    PROFILER_EVENTS.append({'stage': stage, 'elapsed_s': round(float(seconds), 6), **meta})

def page_pevent(page_num, doc_type, stage, strategy, shared_batch_s, batch_size=1,
                tokens_in=0, tokens_out=0, note=None):
    """allocated_s = temps du batch / nb d'éléments : imputation additive,
    pas un temps GPU isolé."""
    batch_size = max(1, int(batch_size or 1))
    row = {'page': page_num, 'doc_type': doc_type, 'stage': stage, 'strategy': strategy,
           'batch_size': batch_size, 'shared_batch_s': round(float(shared_batch_s), 3),
           'allocated_s': round(float(shared_batch_s) / batch_size, 3),
           'tokens_in': int(tokens_in or 0), 'tokens_out': int(tokens_out or 0), 'note': note or ''}
    PAGE_PROFILER_ROWS.append(row)
    _pprint(f"PAGE {page_num} | {doc_type} | {stage} | {strategy} | batch={batch_size} "
            f"partagé={row['shared_batch_s']:.3f}s imputé={row['allocated_s']:.3f}s | "
            f"tokens={row['tokens_in']}+{row['tokens_out']}")

@contextmanager
def pstage(stage, **meta):
    _psync(); t = _pnow()
    try:
        yield
    finally:
        _psync(); d = _pnow() - t
        pevent(stage, d, **meta)
        _pprint(f"{stage} | {d:.3f}s" + (f" | {meta}" if meta else ''))

def profiler_finish(tag):
    """Un jeu de CSV PAR DOSSIER (V13.8.7 écrasait les mêmes fichiers à chaque PDF)."""
    tag = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(tag))[:120]
    if PROFILER_EVENTS:
        df = pd.DataFrame(PROFILER_EVENTS)
        sm = (df.groupby('stage').agg(appels=('stage', 'size'), temps_s=('elapsed_s', 'sum'))
                .reset_index().sort_values('temps_s', ascending=False))
        print('\n================ PROFILER GLOBAL ================')
        print(sm.to_string(index=False))
        gen = df[df.stage == 'MODEL_GENERATE']
        if len(gen) and 'prefill_s' in gen:
            pf = float(gen['prefill_s'].fillna(0).sum()); tot = float(gen['elapsed_s'].sum())
            print(f"MODEL_GENERATE : préfill(+ViT)={pf:.1f}s | décodage={tot-pf:.1f}s | "
                  f"pas de décodage={int(gen['decode_steps'].fillna(0).sum())}")
        df.to_csv(PROFILER_DIR / f'{tag}__events.csv', index=False, encoding='utf-8-sig')
        sm.to_csv(PROFILER_DIR / f'{tag}__summary.csv', index=False, encoding='utf-8-sig')
    if PAGE_PROFILER_ROWS:
        pdf_ = pd.DataFrame(PAGE_PROFILER_ROWS)
        ps = (pdf_.groupby(['page', 'doc_type'], dropna=False)
                  .agg(qwen_etapes=('stage', 'size'), temps_impute_s=('allocated_s', 'sum'),
                       tokens_in=('tokens_in', 'sum'), tokens_out=('tokens_out', 'sum'))
                  .reset_index().sort_values(['page', 'doc_type']))
        print('\n================ RESUME PAR PAGE ================')
        print(ps.to_string(index=False))
        pdf_.to_csv(PROFILER_DIR / f'{tag}__pages_detail.csv', index=False, encoding='utf-8-sig')
        ps.to_csv(PROFILER_DIR / f'{tag}__pages_summary.csv', index=False, encoding='utf-8-sig')

## 7. Inférence GPU

In [ ]:
from transformers import LogitsProcessor, LogitsProcessorList

class _FirstTokenClock(LogitsProcessor):
    """Ne modifie PAS les scores. Note l'instant du 1er pas de décodage
    (= fin du préfill, ViT compris) au prix d'une seule synchronisation CUDA."""
    def __init__(self):
        self.t_first = None; self.steps = 0
    def __call__(self, input_ids, scores):
        if self.t_first is None:
            _psync(); self.t_first = _pnow()
        self.steps += 1
        return scores


def apply_template(messages):
    """Qwen3.x : mode 'thinking' désactivé."""
    try:
        return processor.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def _real_generated_length(ids):
    """Nombre de tokens réellement générés : jusqu'au 1er EOS inclus.
    (En batch, les séquences courtes sont complétées par du padding après l'EOS :
    V13.8.7 comptait ce padding comme des tokens générés.)"""
    ids = ids.tolist()
    for k, tok in enumerate(ids):
        if tok in EOS_TOKEN_IDS:
            return k + 1
    return len(ids)


def ask_batch_mixed(prompts, images, max_new_tokens):
    """Un seul chemin pour batch=1 et batch>1 (mêmes réglages de génération)."""
    if not images:
        return []
    if len(prompts) != len(images):
        raise ValueError('prompts et images doivent avoir la même longueur')
    n = len(images)
    with pstage('CHAT_TEMPLATE', batch=n):
        texts_in = [apply_template([{'role': 'user', 'content': [
            {'type': 'image', 'image': im}, {'type': 'text', 'text': pr}]}])
            for pr, im in zip(prompts, images)]
    with pstage('PROCESSOR', batch=n):
        inputs = processor(text=texts_in, images=list(images), return_tensors='pt', padding=True)
    with pstage('TRANSFER_GPU', batch=n):
        inputs = inputs.to(DEVICE)
    width = int(inputs['input_ids'].shape[1])
    am = inputs.get('attention_mask')
    per_in = am.sum(dim=1).tolist() if am is not None else [width] * n
    total_in = int(sum(per_in))

    clock = _FirstTokenClock() if PROFILE_PREFILL else None
    gen_kwargs = dict(max_new_tokens=int(max_new_tokens), do_sample=False,
                      repetition_penalty=1.0,   # NE PAS changer : pénaliserait « 000 »
                      use_cache=True, pad_token_id=processor.tokenizer.eos_token_id)
    if clock is not None:
        gen_kwargs['logits_processor'] = LogitsProcessorList([clock])
    _psync(); t = _pnow()
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kwargs)
    _psync(); gen_s = _pnow() - t
    if out.shape[0] != n:
        raise RuntimeError(f'Réponses VLM incohérentes : {out.shape[0]} sortie(s) pour {n} image(s)')
    prefill_s = (clock.t_first - t) if (clock and clock.t_first) else None

    gen_cpu = out[:, width:].to('cpu')
    results = []; seq_out = []
    with pstage('DECODE', batch=n):
        for i in range(n):
            tout = _real_generated_length(gen_cpu[i])
            text = processor.decode(gen_cpu[i][:tout], skip_special_tokens=True)
            seq_out.append(tout)
            results.append({'text': text, 'tokens_in': int(per_in[i]), 'tokens_out': tout,
                            'elapsed_s': round(gen_s / n, 3),
                            'hit_max_new_tokens': bool(tout >= int(max_new_tokens))})
    pevent('MODEL_GENERATE', gen_s, batch=n, tokens_in=total_in, tokens_out=int(sum(seq_out)),
           padded_width=width, sequence_tokens_out=str(seq_out), max_new_tokens=int(max_new_tokens),
           prefill_s=None if prefill_s is None else round(prefill_s, 4),
           decode_steps=int(gen_cpu.shape[1]))
    _pprint(f"MODEL_GENERATE | {gen_s:.3f}s | batch={n} in={total_in} (largeur {width}) "
            f"out={seq_out}" + (f" | préfill={prefill_s:.2f}s" if prefill_s is not None else ''))
    del inputs, out
    return results

def is_cuda_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in str(exc).lower()


def run_vlm_chunk(prompts, images, max_new_tokens):
    """Appel GPU SEUL, avec découpage en cas d'OOM. La fusion des résultats se fait
    hors de ce try : une erreur Python de fusion ne relance plus l'inférence et ne
    crée plus de faux 'consensus' par candidats dupliqués."""
    try:
        return ask_batch_mixed(prompts, images, max_new_tokens)
    except Exception as exc:
        if len(images) > 1 and is_cuda_oom(exc):
            print(f'⚠️ OOM batch {len(images)} -> découpage')
            gc.collect(); torch.cuda.empty_cache()
            mid = len(images) // 2
            return (run_vlm_chunk(prompts[:mid], images[:mid], max_new_tokens)
                    + run_vlm_chunk(prompts[mid:], images[mid:], max_new_tokens))
        raise

print('✅ Inférence : tokens réels, préfill/décodage séparés, OOM isolé de la fusion')

## 8. Prompts et schéma
Classification, contrat et contrat spécifique : texte V13.8.7 inchangé.
Engagement : ajout du bloc structurel anti-décalage et de l'objet d'audit `_DOM_EVIDENCE`.

In [ ]:
PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

RÈGLE DE PRIORITÉ ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « جواز العمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
    - une photographie d'identité ;
    - les libellés « Nom » et « Prénom » suivis de valeurs ;
    - les libellés « Date de naissance » / « Lieu de naissance » ;
    - le libellé « Date d'entrée en Algérie » ;
    - des libellés arabes d'identité (اللقب، الإسم، تاريخ الإزدياد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

RÈGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, identifier d'abord le libellé et la structure du formulaire.
3. IMPORTANT — FORMULAIRES PRÉ-IMPRIMÉS :
   la couche des valeurs peut être décalée verticalement par rapport aux libellés.
   Le décalage peut être VERS LE HAUT ou VERS LE BAS et peut concerner TOUTE LA PAGE.
   Ne jamais associer une valeur à un champ uniquement parce qu'elle est exactement
   sur la même ligne horizontale que le libellé.
4. Utiliser conjointement :
   - le libellé ;
   - l'ordre des champs du formulaire ;
   - les libellés voisins au-dessus et au-dessous ;
   - le type de valeur attendu (date, lieu, nom, montant, référence...) ;
   - la cohérence globale de la page.
5. Un décalage global doit rester cohérent sur la page : ne pas déplacer
   arbitrairement une seule valeur d'un champ vers un autre.
6. Conserver la valeur exactement comme elle apparaît : espaces, ponctuation,
   séparateurs, format de date et format de montant.
7. Ne corrige pas l'orthographe.
8. Ne normalise pas les dates.
9. Ne normalise pas les montants.
10. Ne sépare pas automatiquement le nom et le prénom.
11. Ne complète pas une valeur partiellement lisible.
12. N'utilise aucune valeur provenant d'une autre page.
13. Si le libellé est absent, si la valeur est illisible ou si l'association
    libellé/valeur reste ambiguë malgré la structure de la page : retourne null.
14. N'invente jamais une valeur.
15. Retourne uniquement un objet JSON valide, sans commentaire.
16. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

MISE EN PAGE : sur certains engagements, toutes les valeurs imprimées peuvent être
décalées verticalement de façon uniforme vers le haut ou vers le bas par rapport
aux libellés. Utilise la structure entière du formulaire et le type attendu de chaque
valeur ; ne fais jamais un appariement strict par ligne.

Extrais exactement les clés suivantes :

{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :

- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.

- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».

- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».

- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.

- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».

- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».

- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».

- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».

- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de L'Employeur ».

- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de L'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.

- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».

- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».

- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».

- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.

- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.

RÈGLE STRUCTURELLE ANTI-DÉCALAGE — SECTION « 2) Identification de l'opération » :
Sur ce formulaire pré-imprimé, les valeurs sont imprimées en UNE SEULE COUCHE qui
peut être décalée d'environ une ligne vers le HAUT ou vers le BAS par rapport aux
libellés. Une valeur peut alors se trouver AU-DESSUS de son libellé, ou en face du
libellé voisin. Exemple observé : le numéro de contrat imprimé au-dessus de la
ligne « Numéro du contrat », la durée en face de « Numéro du contrat », la date de
début en face de « Duré du contrat », etc.
Les valeurs se déplacent ENSEMBLE et conservent toujours leur ORDRE vertical.
Ne fais donc JAMAIS un appariement par simple alignement horizontal.

Procède en deux temps.

TEMPS 1 — OBSERVER : sans tenir compte des libellés, relève la liste des valeurs
imprimées de la section 2, de HAUT EN BAS, une entrée par valeur (une adresse sur
deux lignes = deux entrées).

TEMPS 2 — ASSOCIER cette liste à l'ordre FIXE du formulaire :
  a. Numéro du contrat            -> nombre entier court
  b. Duré(e) du contrat           -> nombre entier court (mois)
  c. Date de début de contrat     -> date
  d. Date de fin de contrat       -> date
  e. Nom de l'employeur           -> texte (raison sociale)
  f. Adresse de l'employeur       -> texte, 1 ou 2 lignes (une ville isolée à
                                     droite, ex. « ORAN », termine l'adresse)
  g. Salaire net mensuel          -> montant
  h. Montant de la part transférable -> montant
  i. Pourcentage                  -> valeur contenant « % »
  j. Montant domicilié en DZD     -> montant ; souvent ABSENT -> null
Le TYPE de chaque valeur doit être compatible avec son champ : une date ne peut
pas être un numéro de contrat ; une raison sociale ne peut pas être une date ; un
pourcentage ne peut pas être un montant.
Si la liste observée compte une valeur de moins que les champs a..j, c'est le
champ j (Montant domicilié) qui est vide, sauf preuve visuelle contraire.
Ne déplace jamais UNE SEULE valeur isolément : le décalage est commun à la section.
Si l'association reste ambiguë pour un champ, retourne null pour ce champ.

La section « 1) Identification du client » et « Agence domiciliataire » peuvent
subir le même décalage : applique la même logique (nom -> compte -> adresse).

En plus des 15 champs, retourne dans LE MÊME JSON cet objet d'audit :
"_DOM_EVIDENCE": {
  "valeurs_section2_haut_en_bas": [],
  "decalage_vertical": "AUCUN",
  "association_claire": false
}
- "valeurs_section2_haut_en_bas" : la liste du TEMPS 1, recopiée telle que lue.
- "decalage_vertical" : "AUCUN", "HAUT", "BAS" ou "INCERTAIN".
- "association_claire" : true seulement si chaque valeur a trouvé son champ sans
  ambiguïté de type ni d'ordre.
Cet objet décrit uniquement ce qui est visible sur CETTE page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :

{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :

- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».

- CTR_TYPE :
  titre complet du document.

- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTR_DUREE_MOIS :
  valeur après « pour une durée de : »
  et avant « à compter du ».

- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du : ».

- CTR_POSTE :
  valeur après « En qualité de : ».

- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTR_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTR_NATIONALITE :
  valeur après « Nationalité : ».

- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».

- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut : ».

- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net : ».

- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale : ».

- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur : ».

- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».

- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 271901|2026.1|40|00119|DZD
  null si ce cachet est absent de la page.

Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.

- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.

- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :

{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :

- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».

- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.

- CTS_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné : ».

- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité : ».

- CTS_DUREE_MOIS :
  valeur après « pour une durée de : ».

- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du : ».

- CTS_POSTE :
  valeur après « en qualité de : ».

- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) : ».

- CTS_PERE_NOM_PRENOM :
  valeur après « Fils de : »
  et avant « et de : ».

- CTS_MERE_NOM_PRENOM :
  valeur après « et de : ».

- CTS_NATIONALITE :
  valeur après « Nationalité : ».

- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le : »
  et avant « à ».

- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.

- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie : ».

- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle : ».

- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».

- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le : ».

- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « valable du ».

- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---

Cette ligne peut prendre DEUX formes :

  Forme A (salaire inchangé) :
    « Salaire mensuel de base net : 506,471.38 »

  Forme B (augmentation de salaire) :
    « Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « La part transférable : ».

- CTS_PART_PAYABLE_DZD :
  valeur après « La part payable en dinars algérien : ».

- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine : ».

- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie : ».

- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».

- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».

- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.

- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « Le présent contrat a été visé par nous ».
"""

# =====================================================================
# Orientation — prompt générique, valable pour TOUTES les pages
# =====================================================================
PROMPT_PAGE_ORIENTATION = r"""
Cette image est une page scannée d'un dossier administratif : formulaire bancaire,
contrat de travail, ou titre/permis de travail bilingue français/arabe.
La page peut avoir été scannée tournée.
Détermine la rotation HORAIRE qu'il faut appliquer à l'image pour remettre la page
à l'endroit : haut de page en haut, texte français lisible normalement de gauche à
droite.
Réponds STRICTEMENT:
{"rotation_clockwise":0}
Valeurs autorisées uniquement: 0, 90, 180, 270.
Ne lis aucun champ métier et n'invente aucune donnée.
"""

# =====================================================================
# Schéma : 99 champs, ordre et noms STRICTEMENT identiques (hash vérifié plus bas)
# =====================================================================
# DOM / CTR / CTS : les champs sont lus dans le squelette JSON de leur prompt.
# TTR / PTR : l'ancien prompt 21 champs n'est plus envoyé à Qwen (profil TTR-7) ;
# leur liste de champs est donc déclarée explicitement ici.
TTR_SCHEMA_FIELDS = [
    'TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE',
    'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR',
    'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM',
    'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS',
    'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE',
    'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT',
]
PTR_SCHEMA_FIELDS = ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']

PROMPTS_EXTRACTION = {
    'ENGAGEMENT_DOMICILIATION': PROMPT_ENGAGEMENT,
    'CONTRAT_TRAVAIL': PROMPT_CONTRAT,
    'CONTRAT_SPECIFIQUE': PROMPT_CONTRAT_SPECIFIQUE,
}

def champs_attendus_depuis_prompt(prompt):
    match = re.search(r'\{[^{}]*\}', prompt, flags=re.S)
    if not match:
        return []
    try:
        return list(json.loads(match.group(0)).keys())
    except Exception:
        return re.findall(r'"([A-Z0-9_]+)"\s*:', match.group(0))

# L'ordre des clés reproduit celui de V13.8.7.
CHAMPS_ATTENDUS = {
    'ENGAGEMENT_DOMICILIATION': champs_attendus_depuis_prompt(PROMPT_ENGAGEMENT),
    'CONTRAT_TRAVAIL': champs_attendus_depuis_prompt(PROMPT_CONTRAT),
    'CONTRAT_SPECIFIQUE': champs_attendus_depuis_prompt(PROMPT_CONTRAT_SPECIFIQUE),
    'TITRE_TRAVAIL': list(TTR_SCHEMA_FIELDS),
    'PERMIS_TRAVAIL_COUVERTURE': list(PTR_SCHEMA_FIELDS),
}
DOC_TYPES = list(CHAMPS_ATTENDUS)
TYPES_VALIDES = set(DOC_TYPES) | {'AUTRE'}
QWEN_EXTRACTED_TYPES = ('ENGAGEMENT_DOMICILIATION', 'CONTRAT_TRAVAIL', 'CONTRAT_SPECIFIQUE', 'TITRE_TRAVAIL')

# ---------- Profil TTR-7 ----------
TTR7_PROFILE = 'TTR_7_FAST_STRUCTURAL_V2'
TTR7_ACTIVE_FIELDS = ('TTR_NUMERO_PERMIS', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE',
                      'TTR_NATIONALITE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN')
TTR7_INACTIVE_FIELDS = tuple(f for f in TTR_SCHEMA_FIELDS if f not in TTR7_ACTIVE_FIELDS)

# Champs réellement demandés à Qwen : base du taux de remplissage et des critiques.
ACTIVE_FIELDS = {dt: list(f) for dt, f in CHAMPS_ATTENDUS.items()}
ACTIVE_FIELDS['TITRE_TRAVAIL'] = list(TTR7_ACTIVE_FIELDS)
ACTIVE_FIELDS['PERMIS_TRAVAIL_COUVERTURE'] = []

# ---------- Sortie compacte f01, f02... (DOM / CTR / CTS) ----------
COMPACT_ALIAS_BY_DOC = {dt: {f: f'f{i+1:02d}' for i, f in enumerate(CHAMPS_ATTENDUS[dt])}
                        for dt in PROMPTS_EXTRACTION}
COMPACT_FIELD_BY_DOC = {dt: {a: f for f, a in m.items()} for dt, m in COMPACT_ALIAS_BY_DOC.items()}

_COMPACT_AUDIT_LINE = {
    'ENGAGEMENT_DOMICILIATION': '- Conserve aussi, sous son nom exact, l\'objet d\'audit\n  "_DOM_EVIDENCE" demandé plus haut.',
}
_COMPACT_AUDIT_LINE_DEFAULT = ('- Pour TITRE_TRAVAIL uniquement, conserve aussi l\'objet d\'audit\n'
                               '  "TTR_STRUCTURAL_BLOCK" demandé plus haut.')   # texte V13.5 inchangé pour CTR/CTS

def compact_prompt_for_doc(doc_type, prompt):
    mapping = COMPACT_ALIAS_BY_DOC.get(doc_type) or {}
    if not (COMPACT_OUTPUT_ENABLED and mapping):
        return prompt
    match = re.search(r'\{[^{}]*\}', prompt, flags=re.S)
    if not match:
        return prompt
    skeleton = json.dumps({a: None for a in mapping.values()}, ensure_ascii=False, indent=2)
    alias_lines = '\n'.join(f'- {a} = {f}' for f, a in mapping.items())
    audit = _COMPACT_AUDIT_LINE.get(doc_type, _COMPACT_AUDIT_LINE_DEFAULT)
    return prompt[:match.start()] + skeleton + prompt[match.end():] + f"""

FORMAT DE SORTIE COMPACTE — PRIORITÉ ABSOLUE :
Pour réduire la longueur de génération, retourne les champs métier avec les
ALIASES COURTS ci-dessous. Ne retourne PAS les noms longs des champs.

{alias_lines}

Retourne un seul objet JSON valide.
- Chaque alias ci-dessus doit être présent, avec sa valeur ou null.
{audit}
- N'ajoute aucun commentaire ni texte hors JSON.
"""

def expand_compact_extraction(parsed, doc_type):
    """f01/f02/... -> noms canoniques. Les autres clés (objets d'audit) sont conservées."""
    if not isinstance(parsed, dict):
        return {}
    reverse = COMPACT_FIELD_BY_DOC.get(doc_type) or {}
    return {reverse.get(k, k): v for k, v in parsed.items()}

# ---------- Champs critiques ----------
CRITICAL_FIELDS = {
    'ENGAGEMENT_DOMICILIATION': {'DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_NUMERO_CONTRAT',
                                 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT',
                                 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE'},
    'CONTRAT_TRAVAIL': {'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_NUMERO_PERMIS_TRAVAIL',
                        'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_NET'},
    'CONTRAT_SPECIFIQUE': {'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_NUMERO_PERMIS_TRAVAIL',
                           'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS',
                           'CTS_SALAIRE_NET', 'CTS_PART_TRANSFERABLE'},
    # V13.8.7 listait aussi TTR_EMPLOYEUR et TTR_DATE_ENTREE_ALGERIE, jamais extraits en
    # TTR-7 : toute page TTR sortait PARTIELLE / LOW. Critiques = champs ACTIFS uniquement.
    'TITRE_TRAVAIL': {'TTR_NUMERO_PERMIS', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_NOM',
                      'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_NATIONALITE'},
    'PERMIS_TRAVAIL_COUVERTURE': set(),
}
assert all(CRITICAL_FIELDS[dt] <= set(ACTIVE_FIELDS[dt]) for dt in DOC_TYPES)

# ---------- Blocs sensibles au décalage vertical ----------
# Dans un bloc, les valeurs bougent ENSEMBLE : on ne mélange jamais deux lectures.
DOM_SHIFT_BLOCK = ['DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT',
                   'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR',
                   'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE',
                   'DOM_MONTANT_TOTAL_DOMICILIE']
SHIFT_BLOCKS = {
    'ENGAGEMENT_DOMICILIATION': DOM_SHIFT_BLOCK,
    'TITRE_TRAVAIL': ['TTR_DATE_DEBUT', 'TTR_DATE_FIN'],
}

print('✅ Prompts et schéma V14 chargés —', sum(len(v) for v in CHAMPS_ATTENDUS.values()), 'champs')

In [ ]:
# =====================================================================
# TTR-7 — prompt structurel (baseline V13.8.7 conservée mot pour mot hors section permis)
# =====================================================================
_PROMPT_TTR7_HEAD = r"""
Tu lis UNE page TITRE DE TRAVAIL / PERMIS DE TRAVAIL.

OBJECTIF:
extraire UNIQUEMENT ces 7 champs:
TTR_NUMERO_PERMIS, TTR_NOM, TTR_PRENOM, TTR_DATE_NAISSANCE,
TTR_NATIONALITE, TTR_DATE_DEBUT, TTR_DATE_FIN.

REGLES GENERALES:
- Lis uniquement ce qui est visible.
- N'invente jamais une valeur.
- Ne complete jamais un chiffre ou une date.
- N'utilise jamais une autre suite de chiffres de la page pour remplacer une valeur illisible.
- Ne corrige jamais automatiquement O/0, I/1, S/5, B/8, etc.
- Si un champ reste ambigu apres lecture attentive, retourne null.

"""

# Section permis — ancienne consigne V13.8.7 (une des deux références)
_PROMPT_TTR7_PERMIT_LEGACY = r"""TTR_NUMERO_PERMIS — REGLE STRUCTURELLE PRIORITAIRE:
1. Cherche d'abord la ZONE D'EN-TETE du titre/permis, avant le bloc "Le titulaire du present permis..." / avant la description du poste.
2. Le numero de permis est la reference administrative imprimee dans cette zone d'en-tete.
3. Sur les formulaires observes, cette reference peut etre presentee comme DEUX REFERENCES SEPAREES PAR "/".
   Exemple de STRUCTURE uniquement (chiffres fictifs):
   23-00009138 / 31-25-001868
4. La partie utile pour TTR_NUMERO_PERMIS ressemble generalement a:
   NN-NNNNNNNN
   ou NN-NN-NNNNNN
   avec parfois des espaces autour des tirets ou entre certains chiffres.
5. IMPORTANT: une petite suite numerique isolee ailleurs sur la page
   (tampon, manuscrit, code, date, identifiant, numero partiel, etc.)
   N'EST PAS le numero de permis.
6. Ne choisis JAMAIS un candidat uniquement parce qu'il est numerique.
   Il doit etre dans la zone d'en-tete ET avoir la structure d'une reference administrative.
7. Si l'en-tete contient "REFERENCE_A / REFERENCE_B", transcris dans
   TTR_NUMERO_PERMIS la reference qui correspond au format de permis visible.
   Si les deux restent plausibles et que le document ne permet pas de décider avec certitude,
   retourne null plutôt que choisir arbitrairement.
8. Conserve les tirets significatifs. Les espaces typographiques ne sont pas importants.

"""

# Section permis — structure COMPLETE « A / B » (spécification métier)
_PROMPT_TTR7_PERMIT_FULL = r"""TTR_NUMERO_PERMIS — REGLE STRUCTURELLE PRIORITAIRE:
1. Cherche d'abord la ZONE D'EN-TETE du titre/permis, avant le bloc "Le titulaire du present permis..." / avant la description du poste.
2. Le numero de permis est la reference administrative imprimee dans cette zone d'en-tete
   (souvent dans un cadre, en haut a gauche).
3. Cette reference est UNE STRUCTURE COMPLETE en DEUX PARTIES separees par "/":
   NN-NNNNNNNN / NN-NN-NNNNNN
   Exemple de STRUCTURE uniquement (chiffres fictifs):
   23-00009138 / 31-25-001868
   Le "/" FAIT PARTIE de la reference. Ce ne sont PAS deux formats alternatifs:
   transcris TOUJOURS les deux parties, dans l'ordre, separees par " / ".
4. Ignore une eventuelle lettre isolee entre parentheses placee avant la reference,
   par exemple "( R )". Des espaces peuvent entourer les tirets ou le "/".
5. IMPORTANT: une petite suite numerique isolee ailleurs sur la page
   (tampon, manuscrit, code, date, identifiant, numero partiel, numero de serie, etc.)
   N'EST PAS le numero de permis.
6. Ne choisis JAMAIS un candidat uniquement parce qu'il est numerique.
   Il doit etre dans la zone d'en-tete ET avoir la structure complete ci-dessus.
7. Si l'une des deux parties est illisible ou incomplete, retourne null pour
   TTR_NUMERO_PERMIS: jamais une moitie seule, jamais un chiffre devine.
   Reporte dans _TTR7_EVIDENCE ce que tu vois reellement
   (permit_candidat_1 = partie avant "/", permit_candidat_2 = partie apres "/").
8. Conserve les tirets et le "/". Les espaces typographiques ne sont pas importants.

"""

_PROMPT_TTR7_TAIL = r"""ATTENTION AU DECALAGE:
les valeurs peuvent etre decalees verticalement ensemble vers le haut ou le bas.
Le POSTE peut prendre UNE ou DEUX lignes. Ne fais jamais une association par
alignement horizontal fixe.

Pour les dates, utilise la STRUCTURE RELATIVE:
POSTE (1 ou 2 lignes) -> DUREE -> DU/premiere date -> AU/deuxieme date
-> LIEU DE TRAVAIL.
POSTE, DUREE et LIEU sont seulement des ANCRES.
Premiere date apres DUREE = TTR_DATE_DEBUT.
Deuxieme date = TTR_DATE_FIN.
Si ambigu, retourne null. Ne devine pas.

Retourne STRICTEMENT:
{
 "TTR_NUMERO_PERMIS": null,
 "TTR_NOM": null,
 "TTR_PRENOM": null,
 "TTR_DATE_NAISSANCE": null,
 "TTR_NATIONALITE": null,
 "TTR_DATE_DEBUT": null,
 "TTR_DATE_FIN": null,
 "_TTR7_EVIDENCE": {
   "ancre_duree_trouvee": false,
   "ancre_lieu_travail_trouvee": false,
   "poste_lignes": null,
   "date_1_observee": null,
   "date_2_observee": null,
   "duration_text_observe": null,
   "structure_dates_claire": false,
   "permit_zone_entete_trouvee": false,
   "permit_reference_complete_observee": null,
   "permit_candidat_1": null,
   "permit_candidat_2": null,
   "permit_choix_justifie": false
 }
}
"""

PROMPT_TTR7_FAST_STRUCTURAL = (
    _PROMPT_TTR7_HEAD
    + (_PROMPT_TTR7_PERMIT_FULL if TTR_PERMIT_EXPECT_FULL_STRUCTURE else _PROMPT_TTR7_PERMIT_LEGACY)
    + _PROMPT_TTR7_TAIL
)

## 9. Vérification du contrat de schéma

In [ ]:
FIELD_SCHEMA = {k: list(v) for k, v in CHAMPS_ATTENDUS.items()}
_current_hash = hashlib.sha256(json.dumps(FIELD_SCHEMA, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()
assert _current_hash == FIELD_SCHEMA_HASH, (_current_hash, FIELD_SCHEMA_HASH)
assert sum(len(v) for v in FIELD_SCHEMA.values()) == 99
print('✅ Schéma exact vérifié : 99 champs | hash', FIELD_SCHEMA_HASH[:16] + '…')
for k, v in FIELD_SCHEMA.items():
    print(f'   {k:28} : {len(v):2d} champs | actifs Qwen : {len(ACTIVE_FIELDS[k]):2d}')

## 10. Validation déterministe
- **Types** : date, montant, pourcentage, entier court, texte, structure du permis. Jamais de normalisation du RAW.
- **Blocs décalables** : un bloc est *cohérent* si tous ses types sont compatibles, si début < fin, et s'il est conforme à ce que Qwen déclare avoir observé. Les raisons `SOFT:` justifient une relecture sans rendre le bloc incohérent.
- La compatibilité durée/dates du TTR reste une **preuve structurelle locale** (baseline) ; toute autre cohérence appartient à la Partie 2.

In [ ]:
# =====================================================================
# Validation de TYPE (jamais de normalisation, jamais de règle métier)
# =====================================================================
DATE_FIELDS = {
    'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_DATE_SIGNATURE',
    'CTR_DATE_DEBUT_CONTRAT', 'CTR_DATE_NAISSANCE', 'CTR_DATE_DELIVRANCE_PERMIS',
    'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_DATE_SIGNATURE',
    'CTS_DATE_DEBUT_CONTRAT', 'CTS_DATE_NAISSANCE', 'CTS_DATE_DELIVRANCE_PERMIS',
    'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_DATE_DOCUMENT',
    'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_DATE_DELIVRANCE', 'TTR_DATE_NAISSANCE', 'TTR_DATE_ENTREE_ALGERIE',
}
AMOUNT_FIELDS = {
    'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE',
    'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET',
    'CTS_SALAIRE_NET_ANCIEN', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD',
}
# Montants qui doivent être des montants « purs » (bloc décalable DOM).
STRICT_AMOUNT_FIELDS = {'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE'}
# Durées imprimées SANS unité sur le formulaire (« 24 ») : un entier nu est valide.
# V13.8.7 exigeait une unité partout : DOM_DUREE_CONTRAT_MOIS = « 24 » était signalé à tort.
MONTH_COUNT_FIELDS = {'DOM_DUREE_CONTRAT_MOIS', 'CTR_DUREE_MOIS', 'CTS_DUREE_MOIS'}
UNIT_DURATION_FIELDS = {'TTR_DUREE'}
BOOLEAN_FIELDS = {
    'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT',
    'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE',
    'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT',
    'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT',
}
PERMIT_REFERENCE_FIELDS = {'CTR_NUMERO_PERMIS_TRAVAIL', 'CTS_NUMERO_PERMIS_TRAVAIL', 'PTR_NUMERO_SERIE'}
TEXT_FIELDS_REJECT_DATE = {
    'DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE',
    'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR',
    'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR',
    'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_LIEU_PAYS_NAISSANCE',
    'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION',
    'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR',
    'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_LIEU_PAYS_NAISSANCE',
    'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION',
    'TTR_POSTE', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A',
    'TTR_NOM', 'TTR_PRENOM', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION',
    'PTR_WILAYA',
}
# Texte « libre » du bloc décalable DOM : ne doit être ni une date, ni un montant, ni un %.
STRICT_TEXT_FIELDS = {'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR'}

_MONTH_WORDS = ('JANVIER', 'FEVRIER', 'FÉVRIER', 'MARS', 'AVRIL', 'MAI', 'JUIN', 'JUILLET', 'AOUT', 'AOÛT',
                'SEPTEMBRE', 'OCTOBRE', 'NOVEMBRE', 'DECEMBRE', 'DÉCEMBRE')

def _txt(value):
    return '' if value is None else str(value).strip()

def looks_like_date(value):
    s = _txt(value).upper()
    if not s:
        return False
    if re.search(r'\b(?:0?[1-9]|[12]\d|3[01])[./-](?:0?[1-9]|1[0-2])[./-](?:19|20)\d{2}\b', s):
        return True
    if re.search(r'\b(?:19|20)\d{2}[./-](?:0?[1-9]|1[0-2])[./-](?:0?[1-9]|[12]\d|3[01])\b', s):
        return True
    return any(m in s for m in _MONTH_WORDS) and bool(re.search(r'\b(?:19|20)\d{2}\b', s))

def parse_date_for_coherence(value):
    """Uniquement pour comparer deux dates ; ne modifie jamais le RAW."""
    s = _txt(value)
    m = re.search(r'\b(\d{1,2})[./-](\d{1,2})[./-]((?:19|20)\d{2})\b', s)
    if m:
        try: return datetime(int(m.group(3)), int(m.group(2)), int(m.group(1)))
        except Exception: return None
    m = re.search(r'\b((?:19|20)\d{2})[./-](\d{1,2})[./-](\d{1,2})\b', s)
    if m:
        try: return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except Exception: return None
    return None

def _looks_like_pure_amount(s):
    """Chiffres + séparateurs (+ devise éventuelle). Ni date, ni %, ni texte."""
    t = re.sub(r'\b(?:DZD|DA|DINARS?)\b', '', s.upper()).strip()
    return bool(re.fullmatch(r'[\d\s.,;\'’]+', t)) and len(re.sub(r'\D', '', t)) >= 3

def permit_structure_ok(value):
    """Contrôle de STRUCTURE du numéro de permis TTR. Le RAW n'est jamais modifié :
    espaces et préfixe « ( R ) » sont ignorés pour le seul contrôle."""
    c = re.sub(r'\s+', '', _txt(value))
    c = re.sub(r'^\(?[A-Za-z]\)', '', c)
    if TTR_PERMIT_EXPECT_FULL_STRUCTURE:
        return bool(re.fullmatch(r'\d{2}-\d{8}/\d{2}-\d{2}-\d{6}', c))
    return bool(re.fullmatch(r'(?:\d{2}-\d{8}|\d{2}-\d{2}-\d{6})', c))

def semantic_issue_for_field(field, value):
    if is_missing_raw(value):
        return None
    s = _txt(value); digits = re.sub(r'\D', '', s)
    if field in DATE_FIELDS and not looks_like_date(s):
        return 'EXPECTED_DATE'
    if field in AMOUNT_FIELDS and not digits:
        return 'EXPECTED_NUMERIC_VALUE'
    if field in STRICT_AMOUNT_FIELDS and (looks_like_date(s) or '%' in s or not _looks_like_pure_amount(s)):
        return 'EXPECTED_AMOUNT'
    if field == 'DOM_TAUX_TRANSFERABLE':
        if looks_like_date(s) or not digits:
            return 'EXPECTED_PERCENTAGE'
        if '%' not in s:
            m = re.fullmatch(r'\s*(\d{1,3})(?:[.,]\d+)?\s*', s)
            if not (m and int(m.group(1)) <= 100):
                return 'EXPECTED_PERCENTAGE'
    if field == 'DOM_NUMERO_CONTRAT':
        if looks_like_date(s) or '%' in s or not (1 <= len(digits) <= 8) or re.search(r'\d[.,]\d{2}$', s) \
           or len(re.findall(r'[A-Za-zÀ-ÿ]', s)) > 4:
            return 'EXPECTED_CONTRACT_NUMBER'
    if field in MONTH_COUNT_FIELDS:
        if looks_like_date(s) or not re.fullmatch(r'\s*\d{1,3}\s*(?:MOIS|MONTHS?)?\s*\.?', s, flags=re.I):
            return 'EXPECTED_DURATION'
    if field in UNIT_DURATION_FIELDS:
        if not re.search(r'\b\d{1,3}\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES|MOIS|JOUR|JOURS)\b', s.upper()):
            return 'EXPECTED_DURATION'
    if field == 'DOM_COMPTE_LOCAL' and len(digits) < 10:
        return 'EXPECTED_ACCOUNT_NUMBER'
    if field == 'TTR_NUMERO_PERMIS' and not permit_structure_ok(s):
        return 'EXPECTED_PERMIT_STRUCTURE'
    if field in PERMIT_REFERENCE_FIELDS and len(re.sub(r'[^A-Za-z0-9]', '', s)) < 5:
        return 'EXPECTED_REFERENCE'
    if field in BOOLEAN_FIELDS and not isinstance(value, bool) \
       and s.upper() not in {'TRUE', 'FALSE', 'VRAI', 'FAUX', 'OUI', 'NON', '1', '0'}:
        return 'EXPECTED_BOOLEAN'
    if field in TEXT_FIELDS_REJECT_DATE and looks_like_date(s):
        return 'EXPECTED_TEXT_GOT_DATE'
    if field in STRICT_TEXT_FIELDS and ('%' in s or _looks_like_pure_amount(s)
                                        or len(re.findall(r'[A-Za-zÀ-ÿ]', s)) < 3):
        return 'EXPECTED_TEXT'
    return None

_DATE_ORDER_PAIRS = {
    'ENGAGEMENT_DOMICILIATION': [('DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'CONTRACT_DATE_ORDER')],
    'CONTRAT_TRAVAIL': [('CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'PERMIT_DATE_ORDER')],
    'CONTRAT_SPECIFIQUE': [('CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'PERMIT_DATE_ORDER')],
    'TITRE_TRAVAIL': [('TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'WORK_PERMIT_DATE_ORDER')],
}

def coherence_issues_for_data(doc_type, data):
    """Seule contradiction contrôlée en Partie 1 : début > fin (signature d'un décalage).
    Toute autre cohérence (durée, montants, inter-documents) relève de la Partie 2."""
    issues = []
    for f1, f2, code in _DATE_ORDER_PAIRS.get(doc_type, []):
        d1, d2 = parse_date_for_coherence((data or {}).get(f1)), parse_date_for_coherence((data or {}).get(f2))
        if d1 and d2 and d1 >= d2:
            issues.append({'code': code, 'fields': [f1, f2], 'detail': f'{f1}>={f2}'})
    return issues

# ---------- Durée TTR : PREUVE STRUCTURELLE locale (baseline V13.x conservée) ----------
def _duration_months_approx(text):
    if not text:
        return None
    t = str(text).upper().replace(',', ' ')
    y = re.search(r'\b(\d{1,3})\s*(?:AN|ANS|ANNEE|ANNÉE|ANNEES|ANNÉES)\b', t)
    m = re.search(r'\b(\d{1,3})\s*MOIS\b', t)
    d = re.search(r'\b(\d{1,4})\s*(?:JOUR|JOURS)\b', t)
    if not (y or m or d):
        return None
    return (12*int(y.group(1)) if y else 0) + (int(m.group(1)) if m else 0) + (int(d.group(1))/30.44 if d else 0)

def _pair_duration_compatible(start_value, end_value, duration_text):
    d1, d2 = parse_date_for_coherence(start_value), parse_date_for_coherence(end_value)
    if not (d1 and d2 and d1 < d2):
        return False
    dm = _duration_months_approx(duration_text)
    if dm is None:
        return True     # durée illisible : ne doit pas invalider des dates correctes
    return abs((d2 - d1).days / 30.44 - dm) <= 1.5

def _key(value):
    if value is None: return None
    if isinstance(value, bool): return str(value).lower()
    return re.sub(r'\s+', ' ', str(value).strip()).casefold()

def _loose(value):
    return re.sub(r'[\s.]+', '', _txt(value)).casefold()

# =====================================================================
# Cohérence d'un BLOC décalable — contrôles 100 % déterministes
# =====================================================================
def dom_block_report(data, evidence):
    """Raisons d'incohérence du bloc « Identification de l'opération » (liste vide = cohérent)."""
    data = data or {}; why = []
    for f in DOM_SHIFT_BLOCK:
        issue = semantic_issue_for_field(f, data.get(f))
        if issue:
            why.append(f'{f}:{issue}')
    d1 = parse_date_for_coherence(data.get('DOM_DATE_DEBUT_CONTRAT'))
    d2 = parse_date_for_coherence(data.get('DOM_DATE_FIN_CONTRAT'))
    if d1 and d2 and d1 >= d2:
        why.append('DATE_ORDER_INVALID')
    # Un trou AU MILIEU du bloc alors que les champs suivants sont remplis = association douteuse
    # (seul le dernier champ, Montant domicilié, est légitimement vide).
    filled = [not is_missing_raw(data.get(f)) for f in DOM_SHIFT_BLOCK[:-1]]
    if any(filled) and not all(filled):
        why.append('SOFT:GAP_INSIDE_BLOCK:' + ','.join(f for f, ok in zip(DOM_SHIFT_BLOCK[:-1], filled) if not ok))
    # Auto-cohérence avec la liste observée de haut en bas (si Qwen l'a fournie).
    obs = (evidence or {}).get('valeurs_section2_haut_en_bas')
    if isinstance(obs, list) and obs:
        obs_l = [_loose(x) for x in obs if not is_missing_raw(x)]
        pos = -1
        for f in DOM_SHIFT_BLOCK:
            v = _loose(data.get(f))
            if not v:
                continue
            hit = next((i for i in range(max(pos, 0), len(obs_l))
                        if obs_l[i] and (obs_l[i] in v or v in obs_l[i])), None)
            if hit is None:
                why.append(f'SOFT:{f}:NOT_IN_OBSERVED_ORDER'); continue
            pos = hit
    return why

def ttr7_dates_block_report(data, evidence):
    """Raisons d'incohérence du couple DU/AU du titre de travail."""
    data = data or {}; ev = evidence or {}; why = []
    a, b = data.get('TTR_DATE_DEBUT'), data.get('TTR_DATE_FIN')
    d1, d2 = parse_date_for_coherence(a), parse_date_for_coherence(b)
    if not (d1 and d2): why.append('TWO_VALID_DATES_NOT_FOUND')
    elif d1 >= d2: why.append('DATE_ORDER_INVALID')
    if not ev.get('structure_dates_claire'): why.append('DATE_STRUCTURE_NOT_CLEAR')
    if not ev.get('ancre_duree_trouvee'): why.append('DURATION_ANCHOR_NOT_FOUND')
    if not ev.get('ancre_lieu_travail_trouvee'): why.append('WORKPLACE_ANCHOR_NOT_FOUND')
    dur = ev.get('duration_text_observe')
    if d1 and d2 and d1 < d2 and dur and not _pair_duration_compatible(a, b, dur):
        why.append('DURATION_DATE_MISMATCH')
    # Contrôles croisés gratuits : les champs doivent être ce que Qwen dit avoir OBSERVÉ.
    for fld, evk in (('TTR_DATE_DEBUT', 'date_1_observee'), ('TTR_DATE_FIN', 'date_2_observee')):
        o = ev.get(evk)
        if not is_missing_raw(o) and not is_missing_raw(data.get(fld)) and _loose(o) != _loose(data.get(fld)):
            why.append(f'{fld}:DIFFERS_FROM_{evk.upper()}')
    return why

def hard_reasons(reasons):
    """Les raisons 'SOFT:' justifient UNE relecture mais ne rendent pas le bloc incohérent."""
    return [r for r in (reasons or []) if not str(r).startswith('SOFT:')]

def block_rank(reasons):
    """2 = cohérent sans réserve ; 1 = cohérent avec réserve ; 0 = incohérent."""
    return 0 if hard_reasons(reasons) else (1 if reasons else 2)

def shift_block_report(doc_type, data, evidence):
    if doc_type == 'ENGAGEMENT_DOMICILIATION':
        return dom_block_report(data, evidence)
    if doc_type == 'TITRE_TRAVAIL':
        return ttr7_dates_block_report(data, evidence)
    return []

def ttr7_page_report(data, evidence):
    """Validité complète d'une lecture TTR-7 (7 champs + preuves)."""
    data = data or {}; ev = evidence or {}; why = []
    miss = [f for f in TTR7_ACTIVE_FIELDS if is_missing_raw(data.get(f))]
    if miss: why.append('MISSING:' + ','.join(miss))
    p = data.get('TTR_NUMERO_PERMIS')
    if not is_missing_raw(p):
        if not permit_structure_ok(p): why.append('PERMIT_STRUCTURE_INVALID')
        if ev.get('permit_zone_entete_trouvee') is False: why.append('PERMIT_HEADER_ZONE_NOT_FOUND')
        if ev.get('permit_choix_justifie') is False: why.append('PERMIT_CHOICE_NOT_JUSTIFIED')
        ref = ev.get('permit_reference_complete_observee')
        if not is_missing_raw(ref):
            rp, rr = _loose(re.sub(r'^\s*\(?[A-Za-z]\)\s*', '', _txt(p))), _loose(ref)
            if rp not in rr and rr not in rp:
                why.append('PERMIT_DIFFERS_FROM_OBSERVED_REFERENCE')
    for f in ('TTR_DATE_NAISSANCE',):
        if semantic_issue_for_field(f, data.get(f)): why.append(f'{f}:EXPECTED_DATE')
    return why + ttr7_dates_block_report(data, ev)
# =====================================================================
# Bilan d'une lecture
# =====================================================================
def assess_extraction_data(doc_type, data, evidence=None):
    data = data or {}
    active = ACTIVE_FIELDS.get(doc_type) or []
    critical = CRITICAL_FIELDS.get(doc_type, set())
    block = set(SHIFT_BLOCKS.get(doc_type, []))
    semantic = [{'field': f, 'code': c, 'value': data.get(f)}
                for f in active for c in [semantic_issue_for_field(f, data.get(f))] if c]
    coherence = coherence_issues_for_data(doc_type, data)
    critical_missing = sorted(f for f in critical if is_missing_raw(data.get(f)))
    block_why = shift_block_report(doc_type, data, evidence) if block else []
    page_why = ttr7_page_report(data, evidence) if doc_type == 'TITRE_TRAVAIL' else []
    fill = taux_remplissage(data, active)
    sem_trigger = [x['field'] for x in semantic if x['field'] in critical or x['field'] in block]
    trigger = set(critical_missing) | {x['field'] for x in semantic}
    for x in coherence: trigger.update(x['fields'])
    if block_why: trigger.update(block)
    if page_why: trigger.update(TTR7_ACTIVE_FIELDS)
    low_fill = fill < SEUIL_REMPLISSAGE_MIN
    need = bool(ENABLE_PAGE_RECOVERY and (
        (RECOVERY_ON_CRITICAL_MISSING and critical_missing) or
        (RECOVERY_ON_SEMANTIC_MISMATCH and sem_trigger) or
        (RECOVERY_ON_SHIFT_BLOCK_INCOHERENT and (block_why or coherence)) or
        (doc_type == 'TITRE_TRAVAIL' and page_why) or
        (RECOVERY_ON_LOW_FILL and low_fill)))
    return {'critical_missing': critical_missing, 'semantic_issues': semantic, 'coherence_issues': coherence,
            'shift_block_reasons': block_why, 'ttr7_page_reasons': page_why,
            'trigger_fields': sorted(trigger), 'fill_rate': fill, 'low_fill': bool(low_fill),
            'needs_recovery': need,
            'has_problem': bool(critical_missing or semantic or coherence or hard_reasons(block_why)
                                or page_why or low_fill)}

print('✅ Validation V14 : types, blocs décalables, preuves croisées TTR-7')

## 11. Checkpoints

In [ ]:
def canonical_checkpoint_path(pdf_path):
    return JSON_DIR / f'{Path(pdf_path).stem}.json'


def checkpoint_is_complete(dossier, pdf_path):
    if not isinstance(dossier, dict):
        return False
    if (dossier.get('schema_version') != SCHEMA_VERSION or dossier.get('pipeline_version') != PIPELINE_VERSION
            or dossier.get('field_schema_hash') != FIELD_SCHEMA_HASH
            or dossier.get('source_file') != Path(pdf_path).name or not dossier.get('page_records')):
        return False
    try:
        return dossier.get('source_sha256') == sha256_file(pdf_path)
    except Exception:
        return False


def load_existing_checkpoint(pdf_path):
    p = canonical_checkpoint_path(pdf_path)
    if not (RESUME and p.exists()):
        return None
    try:
        d = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        return None
    return d if checkpoint_is_complete(d, pdf_path) else None

## 12. Moteur

In [ ]:
# =====================================================================
# 1) ORIENTATION — toutes les pages, AVANT la classification
# =====================================================================
def _parse_rotation(text):
    obj = parse_json_response(text)
    try:
        a = int(obj.get('rotation_clockwise'))
    except Exception:
        return None
    return a if a in (0, 90, 180, 270) else None


def _ask_orientation(pages, stage):
    imgs = [image_for_classification(p['image']) for p in pages]
    answers = []
    for k in range(0, len(pages), GPU_BATCH_SIZE_CLASSIFICATION):
        chunk = pages[k:k+GPU_BATCH_SIZE_CLASSIFICATION]
        _psync(); t = _pnow()
        outs = run_vlm_chunk([PROMPT_PAGE_ORIENTATION]*len(chunk), imgs[k:k+len(chunk)],
                             ORIENTATION_MAX_NEW_TOKENS)
        _psync(); dt_ = _pnow() - t
        for p, o in zip(chunk, outs):
            page_pevent(p['page_num'], 'NA', stage, 'ORIENTATION_1100', dt_, len(chunk),
                        o['tokens_in'], o['tokens_out'])
            answers.append(o)
    return answers


def _refresh_page_image(pdf_path, page):
    img = standard_image(pdf_path, page)
    page.update({'image': img, 'width': img.width, 'height': img.height})


def _set_rotation(pdf_path, page, angle):
    angle = int(angle) % 360
    if angle != page['rotation_clockwise']:
        page['rotation_clockwise'] = angle
        _refresh_page_image(pdf_path, page)


def detect_page_orientations(pages, pdf_path):
    """Principe « ne pas nuire » : une rotation n'est CONSERVÉE que si Qwen, interrogé sur
    l'image tournée, répond 0. Sinon la page revient à son orientation d'origine et part en revue.

    - Qwen propose 0/90/180/270 (un batch pour toutes les pages).
    - Veto déterministe : un quart de tour proposé sur une page dont les lignes de texte sont
      nettement HORIZONTALES est rejeté sans appel supplémentaire.
    - Re-contrôle des seules pages tournées : rattrape la confusion 90<->270 (270 puis 180 = 90).
    - Si Qwen retombe sur une orientation déjà jugée « pas droite » (ex. 180 puis encore 180),
      il se contredit : arrêt immédiat, retour à 0°, drapeau de revue.
    - Enfin les petites inclinaisons sont mesurées puis corrigées."""
    for p in pages:
        p['orientation'] = {'enabled': bool(ORIENTATION_ENABLED), 'attempts': [], 'flags': [],
                            'tokens_in': 0, 'tokens_out': 0, 'elapsed_s': 0.0, 'calls': 0}
    todo = [p for p in pages if not p.get('is_blank')]
    if ORIENTATION_ENABLED and todo:
        for p in todo:
            p['orientation']['axis_check_original'] = text_axis(p['image'])
        judged = {id(p): set() for p in todo}      # orientations déjà jugées « pas droites »
        confirmed = set()
        pending = list(todo); passes = 0
        max_passes = ORIENTATION_MAX_VERIFY_PASSES if ORIENTATION_VERIFY_ROTATED else 0
        while pending and passes <= max_passes:
            stage = 'ORIENTATION' if passes == 0 else 'ORIENTATION_VERIFY'
            outs = _ask_orientation(pending, stage)
            nxt = []
            for p, o in zip(pending, outs):
                meta = p['orientation']; ans = _parse_rotation(o.get('text', ''))
                meta['attempts'].append({'stage': stage, 'raw_text': o.get('text'), 'answer': ans,
                                         'rotation_before': p['rotation_clockwise']})
                meta['tokens_in'] += o['tokens_in']; meta['tokens_out'] += o['tokens_out']
                meta['elapsed_s'] = round(meta['elapsed_s'] + o['elapsed_s'], 3); meta['calls'] += 1
                if ans is None:
                    meta['flags'].append('ORIENTATION_PARSE_FAILED')
                    if passes == 0: confirmed.add(id(p))          # rien n'a été tourné
                    continue
                if ans == 0:
                    confirmed.add(id(p))
                    if passes > 1: meta['flags'].append('ORIENTATION_CORRECTED_ON_VERIFY')
                    continue
                if (passes == 0 and ans in (90, 270)
                        and meta['axis_check_original']['axis'] == 'HORIZONTAL'):
                    meta['flags'].append('ORIENTATION_QUARTER_TURN_VETOED_BY_AXIS_CHECK')
                    confirmed.add(id(p)); continue
                judged[id(p)].add(p['rotation_clockwise'])
                target = (p['rotation_clockwise'] + ans) % 360
                if target in judged[id(p)] or not ORIENTATION_VERIFY_ROTATED and passes > 0:
                    meta['flags'].append('ORIENTATION_SELF_CONTRADICTION')
                    continue                                       # non confirmée -> retour à 0° plus bas
                _set_rotation(pdf_path, p, target)
                if ORIENTATION_VERIFY_ROTATED:
                    nxt.append(p)
                else:
                    confirmed.add(id(p))
            pending = nxt; passes += 1
        for p in todo:
            if id(p) not in confirmed:
                meta = p['orientation']
                meta['rotation_rejected'] = p['rotation_clockwise']
                _set_rotation(pdf_path, p, 0)
                meta['flags'].append('ORIENTATION_NOT_CONFIRMED_KEPT_ORIGINAL')

    for p in todo:
        meta = p['orientation']
        ax = text_axis(p['image']); meta['axis_check'] = ax
        if ax['axis'] == 'VERTICAL':
            # Après rotation, les lignes devraient être horizontales : Qwen s'est
            # probablement trompé d'un quart de tour. On NE devine PAS le sens : revue.
            meta['flags'].append('ORIENTATION_AXIS_CONFLICT')
        dk = measure_deskew(p['image']); meta['deskew'] = dk
        if dk['deskew_applied']:
            p['deskew_correction_ccw_deg'] = dk['deskew_correction_ccw_deg']
            _refresh_page_image(pdf_path, p)
        meta['rotation_clockwise_applied'] = p['rotation_clockwise']
        meta['review_required'] = bool({'ORIENTATION_AXIS_CONFLICT', 'ORIENTATION_NOT_CONFIRMED_KEPT_ORIGINAL',
                                        'ORIENTATION_PARSE_FAILED'} & set(meta['flags']))
    return pages


# =====================================================================
# 2) CLASSIFICATION — sur l'image redressée
# =====================================================================
def _interpret_classification(parsed):
    dt = parsed.get('type_document') or parsed.get('type') or 'AUTRE'
    try: conf = float(parsed.get('confidence', 0) or 0)
    except Exception: conf = 0.0
    bloc = bool(parsed.get('bloc_identite_present'))
    raw_type = dt if dt in TYPES_VALIDES else 'AUTRE'
    dt = raw_type; requal = False
    if bloc and dt in ('PERMIS_TRAVAIL_COUVERTURE', 'AUTRE'):
        dt = 'TITRE_TRAVAIL'; conf = max(conf, CLASSIFICATION_THRESHOLD); requal = True
    if conf < CLASSIFICATION_THRESHOLD and not requal:
        dt = 'AUTRE'
    return dt, conf, bloc, requal, raw_type


def _new_record(page):
    o = page.get('orientation') or {}
    return {
        'index': page['index'], 'page_num': page['page_num'], 'width': page['width'], 'height': page['height'],
        'white_ratio': page['white_ratio'], 'dark_ratio': page.get('dark_ratio'), 'image': page['image'],
        'rotation_clockwise': page.get('rotation_clockwise', 0),
        'deskew_correction_ccw_deg': page.get('deskew_correction_ccw_deg', 0.0),
        'orientation': o, 'is_blank': bool(page.get('is_blank')),
        'doc_type': 'AUTRE', 'titre_detecte': None, 'bloc_identite_present': False,
        'classification_requalifiee': False, 'classification_retry_fullres': False,
        'classification_confidence': 0.0, 'classification_raw_text': None, 'classification_attempts': [],
        'classification_tokens_in': 0, 'classification_tokens_out': 0, 'classification_elapsed_s': 0.0,
        'raw_data': {}, 'extraction_status': 'NON_LANCEE', 'extraction_error': None,
        'extraction_attempts': [], 'extraction_strategies': [], 'extraction_taux_remplissage': 0.0,
        'extraction_call_count': 0, 'retry_call_count': 0,
        'extraction_tokens_in': 0, 'extraction_tokens_out': 0, 'extraction_elapsed_s': 0.0,
        'field_revisions': [], 'critical_fields_missing': [],
        'quality_flags': list(o.get('flags') or []),
        'field_candidates': {}, 'field_disagreements': [], 'recovery_history': [], 'nullified_values': [],
        'semantic_issues_initial': [], 'semantic_issues_final': [], 'coherence_issues_final': [],
        'page_recovery_triggered': False, 'recovery_passes': 0,
        'field_confidence': {}, 'extraction_confidence_score': 0.0, 'extraction_confidence_band': 'LOW',
        'confidence_method': CONFIDENCE_METHOD, 'is_virtual_subdocument': False,
    }


def _apply_classification(rec, out, strategy):
    parsed = parse_json_response(out.get('text', ''))
    dt, conf, bloc, requal, raw_type = _interpret_classification(parsed)
    rec['classification_attempts'].append({
        'strategy': strategy, 'raw_text': out.get('text'), 'parsed': parsed, 'raw_type': raw_type,
        'resolved_type': dt, 'tokens_in': out.get('tokens_in', 0), 'tokens_out': out.get('tokens_out', 0),
        'elapsed_s': out.get('elapsed_s', 0), 'hit_max_new_tokens': out.get('hit_max_new_tokens', False)})
    rec.update({'doc_type': dt, 'classification_confidence': conf, 'bloc_identite_present': bloc,
                'classification_requalifiee': requal, 'titre_detecte': parsed.get('titre_detecte')})
    att = rec['classification_attempts']
    rec['classification_tokens_in'] = sum(int(a.get('tokens_in', 0) or 0) for a in att)
    rec['classification_tokens_out'] = sum(int(a.get('tokens_out', 0) or 0) for a in att)
    rec['classification_elapsed_s'] = round(sum(float(a.get('elapsed_s', 0) or 0) for a in att), 3)
    rec['classification_raw_text'] = '\n\n'.join(f"[{a['strategy']}] {a.get('raw_text', '')}" for a in att)


def _classify_batch(recs, images, strategy, stage):
    for k in range(0, len(recs), GPU_BATCH_SIZE_CLASSIFICATION):
        chunk = recs[k:k+GPU_BATCH_SIZE_CLASSIFICATION]
        _psync(); t = _pnow()
        outs = run_vlm_chunk([PROMPT_CLASSIFICATION]*len(chunk), images[k:k+len(chunk)],
                             MAX_NEW_TOKENS_CLASSIFICATION)
        _psync(); dt_ = _pnow() - t
        for r, o in zip(chunk, outs):
            _apply_classification(r, o, strategy)
            page_pevent(r['page_num'], r['doc_type'], stage, strategy, dt_, len(chunk),
                        o['tokens_in'], o['tokens_out'])


def classify_pages(pages):
    records = [_new_record(p) for p in pages]
    for r in records:
        if r['is_blank']:
            r['quality_flags'].append('BLANK_PAGE_NOT_SENT_TO_QWEN')
            r['classification_raw_text'] = 'AUCUN_APPEL_QWEN_PAGE_BLANCHE'
    todo = [r for r in records if not r['is_blank']]
    _classify_batch(todo, [image_for_classification(r['image']) for r in todo], 'LOWRES_1100', 'CLASSIFICATION')

    if CLASSIFICATION_RETRY_ON_AUTRE or CLASSIFICATION_RETRY_LOW_CONFIDENCE:
        retry = [r for r in todo if (CLASSIFICATION_RETRY_ON_AUTRE and r['doc_type'] == 'AUTRE')
                 or (CLASSIFICATION_RETRY_LOW_CONFIDENCE
                     and r['classification_confidence'] < CLASSIFICATION_HARD_MIN_CONFIDENCE)]
        first = {id(r): r['classification_attempts'][0]['raw_type'] for r in retry}
        _classify_batch(retry, [r['image'] for r in retry], 'FULLRES_1400_RETRY', 'CLASSIFICATION_RETRY')
        for r in retry:
            r['classification_retry_fullres'] = True
            a, b = first[id(r)], r['classification_attempts'][-1]['raw_type']
            if a != 'AUTRE' and b != 'AUTRE' and a != b:
                # Deux lectures donnent deux types différents : on ne tranche pas.
                r['quality_flags'].append('CLASSIFICATION_CONFLICT')
    return records


def apply_regulatory_classification_gate(records):
    for r in records:
        review = ((r['doc_type'] not in DOC_TYPES)
                  or (r['classification_confidence'] < CLASSIFICATION_HARD_MIN_CONFIDENCE)
                  or ('CLASSIFICATION_CONFLICT' in r['quality_flags']))
        if r['is_blank']:
            review = False
        r['classification_review_required'] = bool(review)
        if review:
            r['quality_flags'].append('CLASSIFICATION_REVIEW_REQUIRED')
        r['quality_flags'] = list(dict.fromkeys(r['quality_flags']))
    return records


def add_virtual_permit_subdocuments(records):
    """PTR : aucun appel Qwen (inchangé). Les 3 champs restent présents à null."""
    nulls = {f: None for f in CHAMPS_ATTENDUS['PERMIS_TRAVAIL_COUVERTURE']}
    physical = [r for r in records if r['doc_type'] == 'PERMIS_TRAVAIL_COUVERTURE']
    for r in physical:
        r['raw_data'] = dict(nulls); r['extraction_status'] = 'DISABLED_V13_7'
        r['quality_flags'] = list(dict.fromkeys(r['quality_flags'] + ['PTR_EXTRACTION_DISABLED_V13_7']))
    if physical:
        return records
    ttr = next((r for r in records if r['doc_type'] == 'TITRE_TRAVAIL'), None)
    if ttr is None:
        return records
    v = _new_record({k: ttr[k] for k in ('index', 'page_num', 'width', 'height', 'white_ratio', 'image')})
    v.update({'doc_type': 'PERMIS_TRAVAIL_COUVERTURE', 'titre_detecte': 'EXTRACTION_DESACTIVEE_V13_7',
              'classification_confidence': ttr['classification_confidence'],
              'classification_raw_text': 'AUCUN_APPEL_QWEN_PTR_V13_7', 'raw_data': dict(nulls),
              'extraction_status': 'DISABLED_V13_7', 'extraction_confidence_score': None,
              'extraction_confidence_band': 'DISABLED', 'quality_flags': ['PTR_EXTRACTION_DISABLED_V13_7'],
              'is_virtual_subdocument': True, 'virtual_parent_doc_type': 'TITRE_TRAVAIL',
              'ptr_extraction_disabled': True, 'classification_review_required': False,
              'rotation_clockwise': ttr['rotation_clockwise'],
              'deskew_correction_ccw_deg': ttr['deskew_correction_ccw_deg']})
    return records + [v]


# =====================================================================
# 3) JOBS D'EXTRACTION
# =====================================================================
def _initial_job(r, pdf_path):
    dt = r['doc_type']
    if dt == 'TITRE_TRAVAIL':
        # Page entière HD 1800, redressée. Logique anti-décalage TTR-7 inchangée.
        return {'record': r, 'prompt': PROMPT_TTR7_FAST_STRUCTURAL,
                'image': page_image(pdf_path, r, IMAGE_MAX_SIZE_HAUTE_DEF, PDF_ZOOM_HAUTE_DEF),
                'strategy': 'TTR7_FAST_STRUCTURAL', 'profile': 'HD', 'mode': 'INITIAL',
                'max_new_tokens': int(MAX_NEW_TOKENS_BY_DOC['TITRE_TRAVAIL'])}
    return {'record': r, 'prompt': compact_prompt_for_doc(dt, PROMPTS_EXTRACTION[dt]), 'image': r['image'],
            'strategy': 'STANDARD', 'profile': 'STANDARD', 'mode': 'INITIAL',
            'max_new_tokens': int(MAX_NEW_TOKENS_BY_DOC.get(dt, MAX_NEW_TOKENS_EXTRACTION))}


_RECOVERY_DOC_HINT = {
    'ENGAGEMENT_DOMICILIATION': """
SPÉCIFIQUE ENGAGEMENT DE DOMICILIATION :
Recommence les deux temps OBSERVER puis ASSOCIER de la section 2. Compte les valeurs
imprimées de haut en bas AVANT de regarder les libellés. Si les valeurs sont décalées
d'une ligne vers le haut ou vers le bas, TOUTES le sont : n'en déplace jamais une seule.
""",
}

def build_page_recovery_prompt(doc_type, report):
    problems = [f'{f}: MISSING' for f in report.get('critical_missing') or []]
    problems += [f"{x['field']}: {x['code']} (lu={x.get('value')!r})" for x in report.get('semantic_issues') or []]
    problems += [f"{x['code']}: {','.join(x['fields'])}" for x in report.get('coherence_issues') or []]
    problems += [f'BLOC DÉCALABLE: {w}' for w in report.get('shift_block_reasons') or []]
    if report.get('low_fill'):
        problems.append(f"LOW_FILL_RATE={report.get('fill_rate')}")
    problem_text = '\n'.join('- ' + p for p in problems) or '- confirmation de la lecture précédente'
    return f"""
RECOVERY PAGE ENTIÈRE — RELECTURE HAUTE DÉFINITION

Relis TOUTE la page de type {doc_type} depuis zéro. Ne relis aucune autre page du PDF.
La lecture précédente a produit au moins une donnée manquante ou incompatible avec
le type de son champ.

IMPORTANT LAYOUT : sur ces formulaires, la couche des valeurs peut être décalée
verticalement de façon globale VERS LE HAUT ou VERS LE BAS. Le même décalage
peut affecter tous les champs de la page. N'associe donc jamais une valeur à un
libellé uniquement par alignement horizontal. Utilise l'ordre du formulaire,
les champs voisins et le TYPE de valeur attendu.

Exemples de contradictions qui doivent être corrigées par relecture :
- un champ DATE contenant une ville, un nom ou un poste ;
- un champ TEXTE contenant une date ;
- date de début postérieure à la date de fin ;
- valeurs décalées d'une ligne à cause de l'impression.
{_RECOVERY_DOC_HINT.get(doc_type, '')}
PROBLÈMES / POINTS À CONFIRMER :
{problem_text}

Tu dois néanmoins réextraire TOUS les champs de cette page afin de disposer du
contexte complet. Ne permute jamais automatiquement des valeurs. Si une valeur
reste ambiguë, retourne null.

{PROMPTS_EXTRACTION[doc_type]}
"""


def _evidence_of(record):
    return record.get('ttr7_evidence') if record['doc_type'] == 'TITRE_TRAVAIL' else record.get('dom_evidence')


def build_recovery_jobs(records, pdf_path):
    jobs = []
    for r in records:
        dt = r['doc_type']
        report = assess_extraction_data(dt, r['raw_data'], _evidence_of(r))
        r['semantic_issues_initial'] = list(report['semantic_issues'])
        r['initial_issue_report'] = report
        if not report['needs_recovery']:
            continue
        r['page_recovery_triggered'] = True
        # La raison du recovery est TOUJOURS tracée (V13.8.7 pouvait laisser une liste vide).
        r['recovery_reasons'] = ([f'MISSING:{f}' for f in report['critical_missing']]
                                 + [f"{x['field']}:{x['code']}" for x in report['semantic_issues']]
                                 + [x['code'] for x in report['coherence_issues']]
                                 + report['shift_block_reasons'] + report['ttr7_page_reasons']
                                 + (['LOW_FILL'] if report['low_fill'] else []))
        r['recovery_reasons'] = list(dict.fromkeys(r['recovery_reasons']))
        img = page_image(pdf_path, r, IMAGE_MAX_SIZE_RECOVERY, PDF_ZOOM_HAUTE_DEF)
        if dt == 'TITRE_TRAVAIL':
            prompt = PROMPT_TTR7_FAST_STRUCTURAL + """
RELECTURE DE SECURITE 2400 PX.
Relis toute la page depuis zero. Ne recopie pas aveuglement la premiere lecture.
"""
            strategy = 'TTR7_RECOVERY_2400_SINGLE'
        else:
            prompt = compact_prompt_for_doc(dt, build_page_recovery_prompt(dt, report))
            strategy = 'PAGE_RECOVERY_2400_SINGLE'
        jobs.append({'record': r, 'prompt': prompt, 'image': img, 'strategy': strategy, 'profile': 'HD',
                     'mode': 'RECOVERY', 'trigger_fields': list(report['trigger_fields']),
                     'max_new_tokens': int(MAX_NEW_TOKENS_RECOVERY_BY_DOC.get(dt, MAX_NEW_TOKENS_RECOVERY))})
    return jobs


# =====================================================================
# 4) FUSION — aucune valeur valide n'est écrasée en silence
# =====================================================================
def _add_field_candidate(record, field, value, strategy):
    if is_missing_raw(value):
        return
    issue = semantic_issue_for_field(field, value)
    record['field_candidates'].setdefault(field, []).append(
        {'value': value, 'strategy': strategy, 'semantic_valid': issue is None, 'semantic_issue': issue})


def _split_parsed(record, parsed):
    """-> (champs canoniques lus, objet de preuve)"""
    dt = record['doc_type']
    parsed = parsed if isinstance(parsed, dict) else {}
    if dt == 'TITRE_TRAVAIL':
        ev = parsed.get('_TTR7_EVIDENCE'); ev = ev if isinstance(ev, dict) else {}
        return {f: parsed.get(f) for f in TTR7_ACTIVE_FIELDS}, ev
    full = expand_compact_extraction(parsed, dt)
    ev = full.get('_DOM_EVIDENCE') if dt == 'ENGAGEMENT_DOMICILIATION' else None
    return {f: full.get(f) for f in CHAMPS_ATTENDUS[dt] if f in full}, (ev if isinstance(ev, dict) else {})


def _store_evidence(record, ev):
    if record['doc_type'] == 'TITRE_TRAVAIL':
        record['ttr7_evidence'] = ev
    elif record['doc_type'] == 'ENGAGEMENT_DOMICILIATION':
        record['dom_evidence'] = ev


def _merge_job_result(job, output):
    record = job['record']; dt = record['doc_type']; mode = job['mode']; strategy = job['strategy']
    data, ev = _split_parsed(record, parse_json_response(output.get('text', '')))
    quality = assess_extraction_data(dt, data, ev)
    attempt = {'strategy': strategy, 'mode': mode, 'is_retry': mode != 'INITIAL',
               'fields_returned': sorted(f for f, v in data.items() if not is_missing_raw(v)),
               'tokens_in': int(output.get('tokens_in', 0) or 0), 'tokens_out': int(output.get('tokens_out', 0) or 0),
               'elapsed_s': float(output.get('elapsed_s', 0) or 0),
               'hit_max_new_tokens': bool(output.get('hit_max_new_tokens')),
               'evidence': ev, 'semantic_issue_count': len(quality['semantic_issues']),
               'critical_missing_count': len(quality['critical_missing']),
               'shift_block_reasons': quality['shift_block_reasons'],
               'ttr7_page_reasons': quality['ttr7_page_reasons'], 'fill_rate': quality['fill_rate']}
    if STORE_QWEN_RAW_TEXT_IN_ATTEMPTS: attempt['raw_text'] = output.get('text')
    if STORE_PARSED_DATA_IN_ATTEMPTS: attempt['parsed_data'] = data
    if attempt['hit_max_new_tokens']:
        record['quality_flags'].append('GENERATION_HIT_MAX_NEW_TOKENS')
    for f, v in data.items():
        _add_field_candidate(record, f, v, strategy)

    raw = record['raw_data']; added = corrected = agreed = disagreed = 0; changed = []

    def _disagree(f, kept, cand, reason):
        record['field_disagreements'].append({'field': f, 'kept': kept, 'candidate': cand,
                                              'strategy': strategy, 'reason': reason})

    def _revise(f, old, new, reason):
        raw[f] = new; changed.append(f)
        record['field_revisions'].append({'field': f, 'old': old, 'new': new, 'strategy': strategy, 'reason': reason})

    if mode == 'INITIAL':
        for f, v in data.items():
            if not is_missing_raw(v):
                raw[f] = v; added += 1
        _store_evidence(record, ev)
    else:
        block = list(SHIFT_BLOCKS.get(dt, []))
        # ---- a) BLOC DÉCALABLE : arbitrage entre lectures ENTIÈRES, jamais champ par champ
        if block:
            old_ev = _evidence_of(record)
            old_why = shift_block_report(dt, raw, old_ev); new_why = shift_block_report(dt, data, ev)
            r_old, r_new = block_rank(old_why), block_rank(new_why)
            same = all(_key(raw.get(f)) == _key(data.get(f)) for f in block)
            attempt['shift_block_arbitration'] = {'old_rank': r_old, 'new_rank': r_new, 'identical': same,
                                                  'old_reasons': old_why, 'new_reasons': new_why}
            if same:
                agreed += sum(not is_missing_raw(raw.get(f)) for f in block)
                if r_new > r_old: _store_evidence(record, ev)
            elif r_new > r_old:
                for f in block:
                    old, new = raw.get(f), data.get(f)
                    if _key(old) != _key(new):
                        _revise(f, old, None if is_missing_raw(new) else new, 'SHIFT_BLOCK_REPLACED_BY_MORE_COHERENT_READ')
                        corrected += 1
                _store_evidence(record, ev)
                record['quality_flags'].append('SHIFT_BLOCK_CORRECTED_BY_RECOVERY')
            else:
                # lecture initiale au moins aussi cohérente : on la garde, l'écart est tracé
                for f in block:
                    if not is_missing_raw(data.get(f)) and _key(raw.get(f)) != _key(data.get(f)):
                        disagreed += 1; _disagree(f, raw.get(f), data.get(f), 'SHIFT_BLOCK_KEPT_INITIAL_READ')
        # ---- b) AUTRES CHAMPS : règle champ par champ
        for f, new in data.items():
            if f in block or is_missing_raw(new):
                continue
            old = raw.get(f); new_issue = semantic_issue_for_field(f, new)
            if is_missing_raw(old):
                if new_issue is None:
                    raw[f] = new; added += 1; changed.append(f)
                continue
            if _key(old) == _key(new):
                agreed += 1; continue
            if semantic_issue_for_field(f, old) is not None and new_issue is None:
                _revise(f, old, new, 'RECOVERY_FIXED_TYPE_INCOMPATIBLE_VALUE'); corrected += 1; continue
            if new_issue is None:
                # deux lectures plausibles et différentes : on ne choisit PAS, on signale
                disagreed += 1; _disagree(f, old, new, 'TWO_VALID_READS_DISAGREE')
        record['retry_call_count'] += 1; record['recovery_passes'] += 1

    attempt.update({'fields_added': added, 'fields_corrected': corrected, 'fields_agreed': agreed,
                    'fields_disagreed': disagreed, 'changed_fields': sorted(set(changed))})
    record['extraction_attempts'].append(attempt)
    record['extraction_call_count'] += 1
    record['extraction_tokens_in'] += attempt['tokens_in']; record['extraction_tokens_out'] += attempt['tokens_out']
    record['extraction_elapsed_s'] = round(record['extraction_elapsed_s'] + attempt['elapsed_s'], 3)
    record['extraction_strategies'].append({'nom': strategy, 'mode': mode, 'champs_ajoutes': added,
                                            'champs_corriges': corrected, 'accords': agreed, 'desaccords': disagreed})
    if mode != 'INITIAL':
        record['recovery_history'].append({
            'strategy': strategy, 'trigger_fields': sorted(job.get('trigger_fields') or []),
            'changed_fields': sorted(set(changed)),
            'issue_report_after': assess_extraction_data(dt, raw, _evidence_of(record))})


def _run_job_chunk(chunk):
    if not chunk:
        return
    _psync(); t = _pnow()
    outs = run_vlm_chunk([j['prompt'] for j in chunk], [j['image'] for j in chunk],
                         max(int(j['max_new_tokens']) for j in chunk))
    _psync(); dt_ = _pnow() - t
    for j, o in zip(chunk, outs):          # fusion HORS du try GPU
        r = j['record']
        try:
            _merge_job_result(j, o)
        except Exception as exc:           # une page en erreur ne bloque pas les autres
            r['extraction_error'] = repr(exc); r['quality_flags'].append('MERGE_ERROR')
            log(f"❌ fusion page {r['page_num']} : {exc!r}")
        page_pevent(r['page_num'], r['doc_type'], j['mode'], j['strategy'], dt_, len(chunk),
                    o.get('tokens_in', 0), o.get('tokens_out', 0))


def run_extraction_jobs(jobs):
    if BATCH_TTR_WITH_STANDARD and all(j['mode'] == 'INITIAL' for j in jobs):
        groups = [(jobs, GPU_BATCH_SIZE_EXTRACTION_STANDARD + GPU_BATCH_SIZE_EXTRACTION_HD)]
    else:
        groups = [([j for j in jobs if j['profile'] == 'STANDARD'], GPU_BATCH_SIZE_EXTRACTION_STANDARD),
                  ([j for j in jobs if j['profile'] != 'STANDARD'], GPU_BATCH_SIZE_EXTRACTION_HD)]
    for group, size in groups:
        for k in range(0, len(group), size):
            _run_job_chunk(group[k:k+size])


# =====================================================================
# 5) FINALISATION / CONFIANCE
# =====================================================================
def _confidence_band(score):
    return 'HIGH' if score >= CONFIDENCE_HIGH else ('MEDIUM' if score >= CONFIDENCE_MEDIUM else 'LOW')


def _compute_field_confidence(record, field, final_report):
    value = record['raw_data'].get(field)
    if is_missing_raw(value):
        return {'score': 0, 'band': 'LOW', 'basis': 'MISSING', 'attempts_supporting': 0, 'distinct_valid_values': 0}
    sem = semantic_issue_for_field(field, value)
    valid = [c for c in record['field_candidates'].get(field, []) if c.get('semantic_valid')]
    key = _key(value)
    # un candidat par STRATÉGIE : une même lecture ne peut pas se soutenir elle-même
    by_strategy = {c['strategy']: _key(c['value']) for c in valid}
    supporting = sum(1 for k in by_strategy.values() if k == key)
    distinct = len({k for k in by_strategy.values() if k is not None})
    revised = any(x['field'] == field for x in record['field_revisions'])
    disagreement = any(x['field'] == field for x in record['field_disagreements'])
    dt = record['doc_type']
    in_block = field in SHIFT_BLOCKS.get(dt, [])
    block_hard = hard_reasons(final_report.get('shift_block_reasons'))
    coherence_fields = {f for x in final_report.get('coherence_issues') or [] for f in x['fields']}
    permit_doubt = (field == 'TTR_NUMERO_PERMIS'
                    and any(str(w).startswith('PERMIT') for w in final_report.get('ttr7_page_reasons') or []))
    block_ok = in_block and not final_report.get('shift_block_reasons')
    if sem is not None or field in coherence_fields:
        score, basis = 25, 'UNRESOLVED_TYPE_OR_DATE_ORDER'
    elif (in_block and block_hard) or permit_doubt:
        score, basis = 40, 'STRUCTURAL_EVIDENCE_UNRESOLVED'
    elif supporting >= 2 and distinct == 1:
        score, basis = (97 if revised else 95), 'CONSENSUS_2_READS'
    elif disagreement or distinct > 1:
        score, basis = 58, 'VALID_VALUES_DISAGREE'
    elif block_ok and not record.get('page_recovery_triggered'):
        score, basis = 93, 'COHERENT_STRUCTURAL_BLOCK_SINGLE_PASS'
    elif revised:
        score, basis = 82, 'RECOVERY_CORRECTED_SINGLE_SUPPORT'
    elif record.get('page_recovery_triggered'):
        score, basis = 84, 'VALID_AFTER_PAGE_RECOVERY'
    else:
        score, basis = 88, 'VALID_SINGLE_PASS'
    return {'score': score, 'band': _confidence_band(score), 'basis': basis,
            'attempts_supporting': supporting, 'distinct_valid_values': distinct, 'semantic_valid': sem is None}


def finalize_extraction_record(record):
    dt = record['doc_type']
    if dt not in DOC_TYPES:
        record.update({'extraction_status': 'NON_APPLICABLE', 'extraction_confidence_score': 0.0,
                       'extraction_confidence_band': 'LOW'})
        return record
    expected = CHAMPS_ATTENDUS[dt]
    for f in expected:
        record['raw_data'].setdefault(f, None)
    if dt == 'PERMIS_TRAVAIL_COUVERTURE' or record.get('extraction_status') == 'DISABLED_V13_7':
        return record
    if record.get('classification_review_required'):
        # Statut historique conservé pour la Partie 2 ; la cause est portée par le drapeau.
        record['extraction_status'] = 'JSON_VIDE'
        record['quality_flags'] = list(dict.fromkeys(record['quality_flags'] + ['EXTRACTION_BLOCKED_CLASSIFICATION_REVIEW']))
        return record
    if dt == 'TITRE_TRAVAIL':
        record.update({'ttr_extraction_profile': TTR7_PROFILE, 'ttr_active_fields': list(TTR7_ACTIVE_FIELDS),
                       'ttr_inactive_fields_not_extracted': list(TTR7_INACTIVE_FIELDS)})

    raw = record['raw_data']; ev = _evidence_of(record)
    report = assess_extraction_data(dt, raw, ev)
    flags = list(record['quality_flags'])

    # Bloc décalable toujours incohérent APRÈS relecture : les champs dont le TYPE est
    # incompatible passent à null (valeur lue conservée). Aucune permutation.
    if (hard_reasons(report['shift_block_reasons']) and record.get('page_recovery_triggered')
            and NULLIFY_TYPE_INCOMPATIBLE_IN_UNRESOLVED_BLOCK):
        for f in SHIFT_BLOCKS.get(dt, []):
            issue = semantic_issue_for_field(f, raw.get(f))
            if issue:
                record['nullified_values'].append({'field': f, 'value_read': raw.get(f), 'reason': issue})
                raw[f] = None
        flags.append('SHIFT_BLOCK_UNRESOLVED')
        report = assess_extraction_data(dt, raw, ev)
        report['shift_block_unresolved'] = True
    if dt == 'TITRE_TRAVAIL':
        record['ttr7_final_valid'] = not report['ttr7_page_reasons']
        record['ttr7_final_reasons'] = report['ttr7_page_reasons']
        if report['ttr7_page_reasons']:
            flags.append('TTR7_FINAL_INVALID')      # V13.8.7 stockait ce verdict sans l'exploiter

    record['extraction_taux_remplissage'] = report['fill_rate']
    record['critical_fields_missing'] = report['critical_missing']
    record['semantic_issues_final'] = report['semantic_issues']
    record['coherence_issues_final'] = report['coherence_issues']
    record['shift_block_reasons_final'] = report['shift_block_reasons']
    if report['critical_missing']: flags.append('CRITICAL_FIELD_MISSING_FINAL')
    if report['semantic_issues']: flags.append('SEMANTIC_FIELD_MISMATCH_FINAL')
    if report['coherence_issues']: flags.append('CROSS_FIELD_COHERENCE_ERROR_FINAL')
    if report['low_fill']: flags.append('LOW_FILL_RATE_FINAL')
    if record.get('page_recovery_triggered'): flags.append('PAGE_RECOVERY_TRIGGERED')
    if record['field_revisions']: flags.append('FIELD_REREAD_BY_PAGE_RECOVERY')
    critical = CRITICAL_FIELDS.get(dt, set()) | set(SHIFT_BLOCKS.get(dt, []))
    blocking = [d for d in record['field_disagreements'] if d['field'] in critical]
    record['blocking_field_disagreements'] = blocking
    if blocking: flags.append('RECOVERY_CRITICAL_FIELD_DISAGREEMENT')
    if len(blocking) < len(record['field_disagreements']): flags.append('RECOVERY_NONCRITICAL_DISAGREEMENT_AUDIT')
    if (record.get('orientation') or {}).get('review_required'): flags.append('ORIENTATION_REVIEW_REQUIRED')
    record['quality_flags'] = list(dict.fromkeys(flags))

    fc = {f: _compute_field_confidence(record, f, report) for f in ACTIVE_FIELDS[dt]}
    for f in expected:
        fc.setdefault(f, {'score': 0, 'band': 'LOW', 'basis': 'FIELD_NOT_IN_ACTIVE_PROFILE',
                          'attempts_supporting': 0, 'distinct_valid_values': 0})
    record['field_confidence'] = fc
    crit = [fc[f]['score'] for f in (CRITICAL_FIELDS.get(dt) or ACTIVE_FIELDS[dt])]
    base = sum(crit) / len(crit) if crit else 0.0
    cls = max(0.0, min(1.0, float(record.get('classification_confidence', 0) or 0))) * 100
    score = round(0.90*base + 0.10*cls, 1)
    problem = report['has_problem'] or report.get('shift_block_unresolved')
    if blocking: score = min(score, 74.0)
    if problem or 'ORIENTATION_REVIEW_REQUIRED' in flags: score = min(score, 69.0)
    record['extraction_confidence_score'] = score
    record['extraction_confidence_band'] = _confidence_band(score)
    if not any(not is_missing_raw(v) for v in raw.values()):
        record['extraction_status'] = 'JSON_VIDE'
    elif problem or blocking:
        record['extraction_status'] = 'PARTIELLE'
    else:
        record['extraction_status'] = 'OK'
    return record


# =====================================================================
# 6) ORCHESTRATION D'UN DOSSIER
# =====================================================================
def extract_classified_pages(records, pdf_path):
    applicable = [r for r in records if r['doc_type'] in QWEN_EXTRACTED_TYPES
                  and not r.get('is_virtual_subdocument')
                  and not (BLOCK_EXTRACTION_ON_CLASSIFICATION_CONFLICT and r.get('classification_review_required'))]
    run_extraction_jobs([_initial_job(r, pdf_path) for r in applicable])
    run_extraction_jobs(build_recovery_jobs(applicable, pdf_path))
    for r in records:
        finalize_extraction_record(r)
    return records


def print_call_diagnostics(records):
    if not PRINT_CALL_DIAGNOSTICS:
        return
    print('\n--- Diagnostic par page ---')
    for r in records:
        print(f"page={r['page_num']} rot={r.get('rotation_clockwise', 0)}° "
              f"deskew={r.get('deskew_correction_ccw_deg', 0)} type={r['doc_type']} "
              f"classif={r['classification_confidence']} appels={r['extraction_call_count']} "
              f"recovery={r.get('recovery_reasons') or '-'} statut={r['extraction_status']} "
              f"conf={r['extraction_confidence_score']}({r['extraction_confidence_band']}) "
              f"flags={r['quality_flags']}")


def _json_safe_record(record):
    return {k: v for k, v in record.items() if k != 'image'}


def process_pdf(pdf_path):
    pdf_path = Path(pdf_path)
    profiler_reset(); render_cache_clear()
    t0 = time.time(); log(f'📁 {pdf_path.name}')
    try:
        with pstage('PDF_TO_PAGES', pdf=pdf_path.name):
            pages = pdf_to_pages(pdf_path)
            for p in pages:
                p['is_blank'] = bool(SKIP_BLANK_PAGES and p['dark_ratio'] <= BLANK_MAX_DARK_RATIO)
        with pstage('ORIENTATION_TOTAL', pages=len(pages)):
            pages = detect_page_orientations(pages, pdf_path)
        with pstage('CLASSIFICATION_TOTAL', pages=len(pages)):
            records = apply_regulatory_classification_gate(classify_pages(pages))
        records = add_virtual_permit_subdocuments(records)
        with pstage('EXTRACTION_TOTAL', pages=len(pages)):
            records = extract_classified_pages(records, pdf_path)
    finally:
        render_cache_clear()
    print_call_diagnostics(records)

    physical = [r for r in records if not r.get('is_virtual_subdocument')]
    def _sum(key): return int(sum(int(r.get(key, 0) or 0) for r in records))
    ori_in = sum(int((r.get('orientation') or {}).get('tokens_in', 0)) for r in physical)
    ori_out = sum(int((r.get('orientation') or {}).get('tokens_out', 0)) for r in physical)
    ori_calls = sum(int((r.get('orientation') or {}).get('calls', 0)) for r in physical)
    tokens_in = _sum('classification_tokens_in') + _sum('extraction_tokens_in') + ori_in
    tokens_out = _sum('classification_tokens_out') + _sum('extraction_tokens_out') + ori_out
    present = sorted({r['doc_type'] for r in physical if r['doc_type'] != 'AUTRE'})
    core = ['ENGAGEMENT_DOMICILIATION', 'CONTRAT_TRAVAIL', 'CONTRAT_SPECIFIQUE', 'TITRE_TRAVAIL']
    cls_calls = sum(len(r['classification_attempts']) for r in records)
    dossier = {
        'schema_version': SCHEMA_VERSION, 'field_schema_hash': FIELD_SCHEMA_HASH, 'field_schema': FIELD_SCHEMA,
        'source_file': pdf_path.name, 'source_sha256': sha256_file(pdf_path),
        'pipeline_version': PIPELINE_VERSION,
        'extraction_engine': {
            'model': 'Qwen3.6-27B-FP8', 'model_path': MODEL_PATH, 'weights_runtime': 'FP8 dequantized -> BF16',
            'classification_policy': 'MANDATORY_VLM_PER_PAGE_NO_PAGE_ORDER',
            'orientation_policy': 'QWEN_ALL_PAGES_BEFORE_CLASSIFICATION+VERIFY_ROTATED+AXIS_CHECK+DESKEW',
            'shift_policy': 'STRUCTURAL_BLOCKS_WHOLE_READ_ARBITRATION_NO_PERMUTATION',
            'ttr_permit_policy': 'FULL_STRUCTURE_A_SLASH_B' if TTR_PERMIT_EXPECT_FULL_STRUCTURE else 'LEGACY_HALF',
            'classification_hard_min_confidence': CLASSIFICATION_HARD_MIN_CONFIDENCE,
            'standard_max_side': IMAGE_MAX_SIZE, 'classification_max_side': IMAGE_MAX_SIZE_CLASSIFICATION,
            'hd_max_side': IMAGE_MAX_SIZE_HAUTE_DEF, 'recovery_max_side': IMAGE_MAX_SIZE_RECOVERY,
            'max_pixels': MAX_PIXELS, 'batch_ttr_with_standard': BATCH_TTR_WITH_STANDARD,
            'recovery_policy': 'SINGLE_WHOLE_PAGE_2400', 'ptr_extraction_policy': 'DISABLED_NO_QWEN_KEEP_3_FIELDS_NULL',
            'confidence_method': CONFIDENCE_METHOD,
            'confidence_note': 'Operational indicator; not native Qwen log-probability',
            'created_at': datetime.now().isoformat(timespec='seconds')},
        'stats': {
            'pages': len(pages), 'page_records': len(records),
            'virtual_subdocuments': sum(bool(r.get('is_virtual_subdocument')) for r in records),
            'pages_rotated': sum(bool(r.get('rotation_clockwise')) for r in physical),
            'pages_deskewed': sum(bool(r.get('deskew_correction_ccw_deg')) for r in physical),
            'orientation_review_required': sum(bool((r.get('orientation') or {}).get('review_required')) for r in physical),
            'blank_pages_skipped': sum(bool(r.get('is_blank')) for r in physical),
            'classification_review_required': sum(bool(r.get('classification_review_required')) for r in records),
            'orientation_calls': ori_calls, 'classification_calls': cls_calls,
            'extraction_calls': _sum('extraction_call_count'), 'retry_calls': _sum('retry_call_count'),
            'qwen_calls_total': ori_calls + cls_calls + _sum('extraction_call_count'),
            'pages_recovered': sum(bool(r.get('page_recovery_triggered')) for r in records),
            'shift_blocks_corrected': sum('SHIFT_BLOCK_CORRECTED_BY_RECOVERY' in r['quality_flags'] for r in records),
            'shift_blocks_unresolved': sum('SHIFT_BLOCK_UNRESOLVED' in r['quality_flags'] for r in records),
            'field_revisions': sum(len(r['field_revisions']) for r in records),
            'field_disagreements': sum(len(r['field_disagreements']) for r in records),
            'pages_confidence_high': sum(r.get('extraction_confidence_band') == 'HIGH' for r in records),
            'pages_confidence_medium': sum(r.get('extraction_confidence_band') == 'MEDIUM' for r in records),
            'pages_confidence_low': sum(r.get('extraction_confidence_band') == 'LOW' for r in records),
            'tokens_in': int(tokens_in), 'tokens_out': int(tokens_out), 'tokens_total': int(tokens_in + tokens_out),
            'elapsed_s': round(time.time() - t0, 3)},
        'page_presence': {
            'present_doc_types': present, 'missing_core_doc_types': [x for x in core if x not in present],
            'classification_review_pages': [r['page_num'] for r in physical if r.get('classification_review_required')],
            'physical_pages': len(pages),
            'note': 'Absence page != champ OCR vide. La Partie 2 exploite cette distinction.'},
        'page_records': [_json_safe_record(r) for r in records],
    }
    canonical_checkpoint_path(pdf_path).write_text(
        json.dumps(dossier, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    profiler_finish(pdf_path.stem)
    return dossier

print('✅ Moteur V14 prêt : orientation -> classification -> extraction -> 1 recovery max -> JSON RAW')

## 13. Tests sans GPU
Ils rejouent notamment vos deux cas réels : engagement décalé d'une ligne vers le haut, titre de travail décalé d'une ligne vers le bas.

In [ ]:
# =====================================================================
# TESTS SANS GPU — tous doivent passer avant l'exécution (« Run All » sûr)
# =====================================================================
# ---- 1. Contrat
assert sum(len(v) for v in CHAMPS_ATTENDUS.values()) == 99
assert len(TTR7_ACTIVE_FIELDS) == 7 and len(TTR7_INACTIVE_FIELDS) == 14
for _dt, _m in COMPACT_ALIAS_BY_DOC.items():
    _fake = {_m[f]: f'V{i}' for i, f in enumerate(CHAMPS_ATTENDUS[_dt])}; _fake['_DOM_EVIDENCE'] = {'x': 1}
    _e = expand_compact_extraction(_fake, _dt)
    assert all(_e[f] == f'V{i}' for i, f in enumerate(CHAMPS_ATTENDUS[_dt])) and '_DOM_EVIDENCE' in _e
assert '_DOM_EVIDENCE' in compact_prompt_for_doc('ENGAGEMENT_DOMICILIATION', PROMPT_ENGAGEMENT)
assert '"f15": null' in compact_prompt_for_doc('ENGAGEMENT_DOMICILIATION', PROMPT_ENGAGEMENT)

# ---- 2. Rotation sans perte + deskew dans le BON sens
_im = Image.fromarray((np.random.RandomState(0).rand(60, 40, 3)*255).astype('uint8'))
for _a in (90, 180, 270):
    assert np.array_equal(np.asarray(rotate_clockwise(rotate_clockwise(_im, _a), 360-_a)), np.asarray(_im))
assert rotate_clockwise(_im, 90).size == (60, 40)
_pg = Image.new('RGB', (1000, 1400), 'white'); _arr = np.asarray(_pg).copy()
for _y in range(150, 1300, 60): _arr[_y:_y+3, 100:900] = 0
_pg = Image.fromarray(_arr)
for _skew in (4.0, -6.0):                      # PIL : angle > 0 = anti-horaire
    _sk = _pg.rotate(_skew, expand=True, fillcolor=(255, 255, 255), resample=Image.BICUBIC)
    _m = measure_deskew(_sk); assert _m['deskew_applied'], _m
    _res, _ = estimate_small_skew_deg(rotate_free_white(_sk, _m['deskew_correction_ccw_deg']))
    assert _res is not None and abs(_res) < 0.6, (_skew, _m, _res)
_hl = cv2.HoughLinesP                                              # OpenCV : (N,1,4) OU (N,4) selon la version
for _shape in ((-1, 1, 4), (-1, 4)):
    cv2.HoughLinesP = lambda *a, _s=_shape, **k: (lambda r: None if r is None else np.asarray(r).reshape(_s))(_hl(*a, **k))
    assert estimate_small_skew_deg(_pg)[0] is not None
cv2.HoughLinesP = _hl
assert not measure_deskew(_pg)['deskew_applied']                      # page droite : image intacte
_arr = np.full((1400, 1000, 3), 255, 'uint8')
for _y in range(150, 1300, 60): _arr[_y:_y+16, 100:900] = 0                  # « lignes de texte » épaisses
_tx = Image.fromarray(_arr)
assert text_axis(_tx)['axis'] == 'HORIZONTAL' and text_axis(rotate_clockwise(_tx, 90))['axis'] == 'VERTICAL'

# ---- 3. Numéro de permis : structure COMPLÈTE
if TTR_PERMIT_EXPECT_FULL_STRUCTURE:
    assert permit_structure_ok('24-00013655 / 31-25-000591')
    assert permit_structure_ok('( R ) 21-00002974 / 31-25-001448')
    assert not permit_structure_ok('24-00013655') and not permit_structure_ok('31-25-000591')
assert not permit_structure_ok('0644005') and not permit_structure_ok('7120')
assert semantic_issue_for_field('TTR_NUMERO_PERMIS', '0644005') == 'EXPECTED_PERMIT_STRUCTURE'

# ---- 4. Types
assert semantic_issue_for_field('DOM_DUREE_CONTRAT_MOIS', '24') is None          # faux positif V13.8.7 corrigé
assert semantic_issue_for_field('CTR_DUREE_MOIS', '24 Mois') is None
assert semantic_issue_for_field('DOM_DUREE_CONTRAT_MOIS', '27/09/2025') == 'EXPECTED_DURATION'
assert semantic_issue_for_field('TTR_DUREE', 'TAGE / LAMINOIR - 2') == 'EXPECTED_DURATION'
assert semantic_issue_for_field('TTR_DATE_DEBUT', 'ORAN') == 'EXPECTED_DATE'
for _v in ('371.099.33', '8 461 064,88', '506471;38', '14 326 077.60'):
    assert semantic_issue_for_field('DOM_SALAIRE_NET_MENSUEL', _v) is None, _v
assert semantic_issue_for_field('DOM_PART_TRANSFERABLE', '95%') == 'EXPECTED_AMOUNT'
assert semantic_issue_for_field('DOM_TAUX_TRANSFERABLE', '14 326 077.60') == 'EXPECTED_PERCENTAGE'
assert semantic_issue_for_field('DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', '628.336.74') == 'EXPECTED_TEXT'

# ---- 5. Engagement : cas RÉEL décalé d'une ligne vers le HAUT (photo fournie)
_DOM_OK = {'DOM_NUMERO_CONTRAT': '1302', 'DOM_DUREE_CONTRAT_MOIS': '24', 'DOM_DATE_DEBUT_CONTRAT': '27/09/2025',
           'DOM_DATE_FIN_CONTRAT': '26/09/2027', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR': 'SPA TOSYALI IRON STEEL ALGERIE',
           'DOM_ADRESSE_EMPLOYEUR': 'POLE ECONOMIQUE PLATEAU GOURIRATE COMMUNE BETHIOUA ORAN',
           'DOM_SALAIRE_NET_MENSUEL': '628.336.74', 'DOM_PART_TRANSFERABLE': '596 919.90',
           'DOM_TAUX_TRANSFERABLE': '95%', 'DOM_MONTANT_TOTAL_DOMICILIE': '14 326 077.60'}
_vals = list(_DOM_OK.values())
_DOM_SHIFTED = dict(zip(DOM_SHIFT_BLOCK, _vals[1:] + [None]))      # lecture naïve par alignement horizontal
_EV = {'valeurs_section2_haut_en_bas': ['1302', '24', '27/09/2025', '26/09/2027', 'SPA TOSYALI IRON STEEL ALGERIE',
       'POLE ECONOMIQUE PLATEAU GOURIRATE COMMUNE BETHIOUA', 'ORAN', '628.336.74', '596 919.90', '95%', '14 326 077.60']}
assert dom_block_report(_DOM_OK, _EV) == []
assert hard_reasons(dom_block_report(_DOM_SHIFTED, _EV))
assert block_rank(dom_block_report(_DOM_OK, _EV)) == 2 > block_rank(dom_block_report(_DOM_SHIFTED, _EV))
_no_total = dict(_DOM_OK, DOM_MONTANT_TOTAL_DOMICILIE=None)                       # champ légitimement vide
assert dom_block_report(_no_total, {}) == []

# ---- 6. Fusion : le bloc est remplacé EN ENTIER par la lecture cohérente, sans mélange
def _fake_record(dt):
    r = _new_record({'index': 0, 'page_num': 1, 'width': 990, 'height': 1400, 'white_ratio': 0.8, 'image': None})
    r.update({'doc_type': dt, 'classification_confidence': 0.98, 'classification_review_required': False}); return r
def _dom_json(block, ev):
    d = {'DOM_NOM_RAISON_SOCIAL_CLIENT': 'MOHAMMED KHAJA NASEERUDDIN', 'DOM_COMPTE_LOCAL': '02700731010929900170',
         'DOM_ADRESSE_CLIENT': 'BASE DE VIE TOSYALI', 'DOM_AGENCE_DOMICILIATAIRE': 'MOSTAGANEM',
         'DOM_DATE_SIGNATURE': '01/12/2025', **block}
    a = COMPACT_ALIAS_BY_DOC['ENGAGEMENT_DOMICILIATION']
    return json.dumps({**{a[k]: v for k, v in d.items()}, '_DOM_EVIDENCE': ev})
_r = _fake_record('ENGAGEMENT_DOMICILIATION')
_merge_job_result({'record': _r, 'mode': 'INITIAL', 'strategy': 'STANDARD'}, {'text': _dom_json(_DOM_SHIFTED, {})})
_rep = assess_extraction_data('ENGAGEMENT_DOMICILIATION', _r['raw_data'], _r.get('dom_evidence'))
assert _rep['needs_recovery'] and _r['raw_data']['DOM_NUMERO_CONTRAT'] == '24'
_merge_job_result({'record': _r, 'mode': 'RECOVERY', 'strategy': 'PAGE_RECOVERY_2400_SINGLE',
                   'trigger_fields': _rep['trigger_fields']}, {'text': _dom_json(_DOM_OK, _EV)})
assert {f: _r['raw_data'][f] for f in DOM_SHIFT_BLOCK} == _DOM_OK
finalize_extraction_record(_r)
assert _r['extraction_status'] == 'OK' and 'SHIFT_BLOCK_CORRECTED_BY_RECOVERY' in _r['quality_flags'], _r['quality_flags']

# bloc toujours décalé après relecture -> champs de type incompatible à null, valeur lue conservée
_r = _fake_record('ENGAGEMENT_DOMICILIATION'); _r['page_recovery_triggered'] = True
_merge_job_result({'record': _r, 'mode': 'INITIAL', 'strategy': 'STANDARD'}, {'text': _dom_json(_DOM_SHIFTED, {})})
finalize_extraction_record(_r)
assert _r['raw_data']['DOM_DATE_FIN_CONTRAT'] is None and _r['extraction_status'] == 'PARTIELLE'
assert any(x['field'] == 'DOM_DATE_FIN_CONTRAT' and x['value_read'].startswith('SPA') for x in _r['nullified_values'])

# ---- 7. TTR-7 : lecture parfaite = OK/HIGH ; deux lectures valides divergentes = jamais d'écrasement
_EV7 = {'ancre_duree_trouvee': True, 'ancre_lieu_travail_trouvee': True, 'structure_dates_claire': True,
        'permit_zone_entete_trouvee': True, 'permit_choix_justifie': True, 'duration_text_observe': '2 ANS, 0 JOURS',
        'date_1_observee': '14/04/2025', 'date_2_observee': '13/04/2027',
        'permit_reference_complete_observee': '24-00013655 / 31-25-000591'}
_T = {'TTR_NUMERO_PERMIS': '24-00013655 / 31-25-000591' if TTR_PERMIT_EXPECT_FULL_STRUCTURE else '24-00013655',
      'TTR_NOM': 'INANC', 'TTR_PRENOM': 'MEVLUT', 'TTR_DATE_NAISSANCE': '01/12/1977', 'TTR_NATIONALITE': 'TURQUE',
      'TTR_DATE_DEBUT': '14/04/2025', 'TTR_DATE_FIN': '13/04/2027'}
_r = _fake_record('TITRE_TRAVAIL')
_merge_job_result({'record': _r, 'mode': 'INITIAL', 'strategy': 'TTR7_FAST_STRUCTURAL'},
                  {'text': json.dumps({**_T, '_TTR7_EVIDENCE': _EV7})})
assert not assess_extraction_data('TITRE_TRAVAIL', _r['raw_data'], _r['ttr7_evidence'])['needs_recovery']
finalize_extraction_record(_r)
assert _r['extraction_status'] == 'OK' and _r['extraction_confidence_band'] == 'HIGH', _r['extraction_confidence_score']
assert len(_r['raw_data']) == 21 and all(_r['raw_data'][f] is None for f in TTR7_INACTIVE_FIELDS)

# cas RÉEL décalé vers le BAS (photo fournie) lu naïvement : Du = « 2 ANS, 0 JOURS » -> relecture
_bad = dict(_T, TTR_DATE_DEBUT='2 ANS, 0 JOURS', TTR_DATE_FIN='14/04/2025')
assert 'TWO_VALID_DATES_NOT_FOUND' in ttr7_dates_block_report(_bad, _EV7)
# champs ≠ preuves observées -> relecture
assert any('DIFFERS_FROM' in w for w in ttr7_dates_block_report(dict(_T, TTR_DATE_FIN='18/04/2027'), _EV7))

_r = _fake_record('TITRE_TRAVAIL')
_merge_job_result({'record': _r, 'mode': 'INITIAL', 'strategy': 'TTR7_FAST_STRUCTURAL'},
                  {'text': json.dumps({**dict(_T, TTR_NOM=None), '_TTR7_EVIDENCE': _EV7})})
_p2 = _T['TTR_NUMERO_PERMIS'][:-1] + '7'
_merge_job_result({'record': _r, 'mode': 'RECOVERY', 'strategy': 'TTR7_RECOVERY_2400_SINGLE', 'trigger_fields': []},
                  {'text': json.dumps({**dict(_T, TTR_NUMERO_PERMIS=_p2), '_TTR7_EVIDENCE': dict(_EV7, permit_reference_complete_observee=_p2)})})
assert _r['raw_data']['TTR_NOM'] == 'INANC'                                   # champ manquant complété
assert _r['raw_data']['TTR_NUMERO_PERMIS'] == _T['TTR_NUMERO_PERMIS']        # PAS d'écrasement silencieux
finalize_extraction_record(_r)
assert _r['extraction_status'] == 'PARTIELLE' and _r['field_confidence']['TTR_NUMERO_PERMIS']['score'] <= 58

# ---- 8. Un numéro isolé (n° de série de la couverture) n'est plus accepté comme permis
_r = _fake_record('TITRE_TRAVAIL')
_merge_job_result({'record': _r, 'mode': 'INITIAL', 'strategy': 'TTR7_FAST_STRUCTURAL'},
                  {'text': json.dumps({**dict(_T, TTR_NUMERO_PERMIS='0644005'), '_TTR7_EVIDENCE': _EV7})})
finalize_extraction_record(_r)
assert _r['extraction_status'] == 'PARTIELLE' and 'TTR7_FINAL_INVALID' in _r['quality_flags']

# ---- 9. Classification
assert _interpret_classification({'type_document': 'PERMIS_TRAVAIL_COUVERTURE', 'confidence': 0.5,
                                  'bloc_identite_present': True})[0] == 'TITRE_TRAVAIL'
assert _interpret_classification({'type_document': 'CONTRAT_TRAVAIL', 'confidence': 0.7})[0] == 'AUTRE'
assert _parse_rotation('{"rotation_clockwise":270}') == 270 and _parse_rotation('{"rotation_clockwise":45}') is None
print('✅ Tous les tests hors GPU passent (contrat, rotation, deskew, permis, types, décalage DOM/TTR, fusion)')

## 14. Exécution

In [ ]:
if not pdfs:
    print('⚠️ Aucun PDF dans', INPUT_DIR)
else:
    results = []; errors = []
    for i, pdf in enumerate(pdfs, 1):
        print(f'\n[{i}/{len(pdfs)}] {pdf.name}')
        existing = load_existing_checkpoint(pdf)
        if existing is not None:
            print('↪ checkpoint RAW valide réutilisé'); d = existing; statut = 'REPRIS'
        else:
            try:
                d = process_pdf(pdf); statut = 'TRAITE'
            except Exception as exc:
                log(f'❌ {pdf.name} : {exc!r}')
                errors.append({'source_file': pdf.name, 'error': repr(exc),
                               'date': datetime.now().isoformat(timespec='seconds')})
                continue
        s = d.get('stats') or {}
        results.append({'source_file': d.get('source_file'), 'status': statut, 'sha256': d.get('source_sha256'),
                        **{k: s.get(k) for k in ('pages', 'pages_rotated', 'pages_deskewed', 'qwen_calls_total',
                                                 'orientation_calls', 'classification_calls', 'extraction_calls',
                                                 'retry_calls', 'pages_recovered', 'shift_blocks_corrected',
                                                 'shift_blocks_unresolved', 'field_disagreements',
                                                 'pages_confidence_high', 'pages_confidence_medium',
                                                 'pages_confidence_low', 'tokens_in', 'tokens_out', 'elapsed_s')},
                        'json_path': str(canonical_checkpoint_path(pdf))})
        print(f"✅ {statut} | pages={s.get('pages')} (tournées={s.get('pages_rotated')}) | appels Qwen={s.get('qwen_calls_total')} "
              f"| recoveries={s.get('pages_recovered')} | blocs corrigés={s.get('shift_blocks_corrected')} "
              f"| tokens={s.get('tokens_total')} | {s.get('elapsed_s')}s")

    MANIFEST_PATH.write_text(json.dumps({
        'schema_version': SCHEMA_VERSION, 'pipeline_version': PIPELINE_VERSION, 'field_schema_hash': FIELD_SCHEMA_HASH,
        'generated_at': datetime.now().isoformat(timespec='seconds'), 'results': results, 'errors': errors},
        ensure_ascii=False, indent=2), encoding='utf-8')
    pd.DataFrame(results).to_csv(INDEX_CSV_PATH, index=False, encoding='utf-8-sig')
    print('\n✅ Manifest :', MANIFEST_PATH); print('✅ Index    :', INDEX_CSV_PATH)
    print('✅ JSON RAW :', JSON_DIR);       print('✅ Profiler :', PROFILER_DIR)

## 15. Lecture du JSON RAW — nouveautés d'audit

Par page : `rotation_clockwise`, `deskew_correction_ccw_deg`, `orientation` (réponses Qwen, contrôle d'axe, drapeaux), `recovery_reasons` (jamais vide si relecture), `dom_evidence` / `ttr7_evidence`, `extraction_attempts[*].shift_block_arbitration`, `nullified_values`, `ttr7_final_valid` + `ttr7_final_reasons`.

Drapeaux à surveiller en Partie 2 / revue humaine :
`ORIENTATION_REVIEW_REQUIRED`, `ORIENTATION_CORRECTED_ON_VERIFY`, `CLASSIFICATION_CONFLICT`, `SHIFT_BLOCK_CORRECTED_BY_RECOVERY`, `SHIFT_BLOCK_UNRESOLVED`, `RECOVERY_CRITICAL_FIELD_DISAGREEMENT`, `TTR7_FINAL_INVALID`, `GENERATION_HIT_MAX_NEW_TOKENS`.

## 16. Avant mise en production

1. Cellule 4 : vérifier `max_pixels` effectif et la présence du fast path DeltaNet.
2. Passer le corpus A/B (page normale, 90°/180°/270°, inclinée 3–10°, TTR décalé, POSTE sur 2 lignes, engagement décalé haut **et** bas, scan dégradé, dossier avec/sans recovery, page passeport, page blanche) et comparer **champ par champ** avec V13.8.7.
3. Seulement ensuite, tester `BATCH_TTR_WITH_STANDARD = True` (gain attendu : ~330 pas de décodage séquentiels en moins par dossier).